In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║                  NENA — Sign-to-Swahili Voice App                   ║
║         Multimodal Contrastive Framework · TSL/KSL Edition          ║
║                                                                      ║
║  USAGE: Paste each cell into Google Colab (GPU runtime).             ║
║  Requires: Runtime → Change runtime type → T4 GPU                   ║
╚══════════════════════════════════════════════════════════════════════╝

STAGE OVERVIEW
--------------
  Cell 1 — Install dependencies
  Cell 2 — Imports & GPU check
  Cell 3 — CLIP model loader
  Cell 4 — Swahili sign vocabulary (placeholder — replace with real TSL data)
  Cell 5 — CLIP embedding pipeline
  Cell 6 — Few-shot sign classifier
  Cell 7 — TTS voice router (Male / Female Swahili neural voice)
  Cell 8 — Full end-to-end demo
  Cell 9 — Webcam / video file inference loop
  Cell 10 — Save checkpoint & push to GitHub instructions
"""

GITHUB_INSTRUCTIONS = """
╔══════════════════════════════════════════════════════════╗
║          Push Nena to GitHub from Colab                  ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  1. File → Save a copy in GitHub                         ║
║     (easiest, no terminal needed)                        ║
║                                                          ║
║  OR via terminal:                                        ║
║                                                          ║
║  !git config --global user.email "you@example.com"       ║
║  !git config --global user.name "Your Name"              ║
║  !git clone https://github.com/YOUR_USERNAME/nena.git    ║
║  !cp /content/*.py /content/nena/                        ║
║  !cp /content/nena_prototypes.pt /content/nena/          ║
║  %cd /content/nena                                       ║
║  !git add .                                              ║
║  !git commit -m "Add Nena sign-to-Swahili pipeline"      ║
║  !git push                                               ║
║                                                          ║
║  Repo structure to aim for:                              ║
║    nena/                                                 ║
║    ├── nena_sign_to_swahili.py   ← this file             ║
║    ├── nena_prototypes.pt        ← saved embeddings      ║
║    ├── data/                     ← your TSL frames       ║
║    │   ├── habari/               ← one folder per sign   ║
║    │   ├── asante/                                       ║
║    │   └── ...                                           ║
║    └── README.md                                         ║
╚══════════════════════════════════════════════════════════╝

NEXT STEPS TO COLLECT TSL DATA
───────────────────────────────
1. Film 10–20 short clips of each sign (phone camera is fine).
2. Extract frames:
     !ffmpeg -i my_sign.mp4 -vf fps=5 data/habari/frame_%04d.jpg
3. Add frame paths to the VOCABULARY list in Cell 4:
     "reference_img": ["data/habari/frame_0001.jpg", ...]
4. Re-run Cell 5 to rebuild prototypes with real visual signal.
5. Accuracy will improve dramatically with even 5 real frames per sign.

DATASETS TO WATCH (from Awesome-Sign-Language repo):
  · How2Sign (ASL) — structure your TSL data the same way
  · OpenASL — study the annotation format
  · Phoenix-2014T — good gloss-text alignment reference
"""

print(GITHUB_INSTRUCTIONS)


╔══════════════════════════════════════════════════════════╗
║          Push Nena to GitHub from Colab                  ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  1. File → Save a copy in GitHub                         ║
║     (easiest, no terminal needed)                        ║
║                                                          ║
║  OR via terminal:                                        ║
║                                                          ║
║  !git config --global user.email "you@example.com"       ║
║  !git config --global user.name "Your Name"              ║
║  !git clone https://github.com/YOUR_USERNAME/nena.git    ║
║  !cp /content/*.py /content/nena/                        ║
║  !cp /content/nena_prototypes.pt /content/nena/          ║
║  %cd /content/nena                                       ║
║  !git add .                                              ║
║  !git commit -m "Add 

# 1 — Install dependencies

In [ ]:
!pip install -q open-clip-torch torch torchvision Pillow
!pip install -q google-cloud-texttospeech
!pip install -q opencv-python-headless
!pip install -q gtts playsound
!apt-get -qq install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


# 2 — Imports & GPU check

In [ ]:
import torch
import open_clip
import numpy as np
from PIL import Image
import cv2
import os
import json
from pathlib import Path
from typing import Optional

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

✅ Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


# 3 — Load CLIP model

In [ ]:
print("Loading CLIP model… (first run downloads ~350 MB)")

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"   # Strong multilingual pretraining
)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

print("✅ CLIP ViT-B/32 loaded.")

Loading CLIP model… (first run downloads ~350 MB)


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

✅ CLIP ViT-B/32 loaded.


# 4 — Swahili sign vocabulary

### This is your PROTOTYPE vocabulary.
### Each entry = one sign concept with:
###   - swahili_text  : what gets spoken aloud
###   - description   : English description of the sign gesture
###                     (used to build CLIP text prototypes)
###   - reference_img : path to a reference image (None = text-only prototype)
### HOW TO EXPAND:
###   1. Record 5–10 video clips of each sign.
###   2. Extract key frames → save as JPG.
###   3. Add the frame paths to reference_img lists.
###   4. Run Cell 5 to rebuild embeddings.


In [ ]:
VOCABULARY = [
    # ── Greetings / Courtesy (previously described, kept as-is) ──────────
    {
        "id": "habari",
        "swahili_text": "Habari",
        "description": "A person holding the left hand flat and open at chest level while the right flat hand rests on top of it, then lifts the right hand upward and forward away from the chest in a presenting or offering motion",
        "reference_img": None,
    },
    {
        "id": "nzuri",
        "swahili_text": "Nzuri",
        "description": "A person raising both open hands to shoulder height with all fingers spread wide apart, then curling all fingers downward to close both hands into loose fists simultaneously as a sign meaning good or fine",
        "reference_img": None,
    },
    {
        "id": "asante",
        "swahili_text": "Asante",
        "description": "A person raises both hands together in front of the chest with the fingertips of each hand touching and pointing upward near the chin, then bows the head slightly forward while lowering both hands down to rest at chest level with fingers still loosely pressed together, as a gesture of gratitude",
        "reference_img": None,
    },
    {
        "id": "mbaya",
        "swahili_text": "Mbaya",
        "description": "A person starts with both hands relaxed near the waist, then raises the right hand up to shoulder/head height, forming a loose fist with the thumb pointing downward and shaking it slightly from side to side while the left hand remains lowered near the stomach",
        "reference_img": None,
    },
    {
        "id": "maji",
        "swahili_text": "Maji",
        "description": "A person holds the left hand open and flat facing upward at chest level, while the right hand, with only the index and middle fingers extended together and the other fingers curled in, taps the fingertips downward onto the left palm two or three times before both hands lower to a relaxed position",
        "reference_img": None,
    },
    {
        "id": "tafadhali_au_samahani",
        "swahili_text": "Tafadhali au Samahani",
        "description": "A person presses both open hands together in front of the chest with fingers spread, rubbing the palms together, then raises the right hand upward to shoulder height with the palm facing outward in an open, flat position; the right hand then relaxes into a pinched fingertip shape and circles gently near the cheek before both hands come back down, with the right hand (fingers drawn together) resting on top of the open left palm at chest height",
        "reference_img": None,
    },
    {
        "id": "chakula",
        "swahili_text": "Chakula",
        "description": "A person brings the right hand up to the mouth with the fingertips and thumb pinched together as if holding food, tapping or bobbing the pinched fingertips toward the lips two or three times while the left hand rests lower near the stomach",
        "reference_img": None,
    },
    {
        "id": "jina_langu",
        "swahili_text": "Jina langu",
        "description": "A person extends the index fingers of both hands and taps the fingertips together repeatedly in front of the chest (sign for 'name'), then brings the right hand to rest flat and open against the chest (sign for 'my/mine'), ending with the fingertips of both hands drawn together again near the chest",
        "reference_img": None,
    },
    {
        "id": "kwaheri",
        "swahili_text": "Kwaheri",
        "description": "A person starts with both hands relaxed near the waist, then raises the right hand to shoulder/head height with fingers open and waves it side to side several times in a standard goodbye-wave motion, while the left hand stays lowered near the stomach",
        "reference_img": None,
    },

    # ── Everyday action verbs (new signs, S0010-S0016) ────────────────────
    {
        # kula.mp4 — right hand held in a loose fist near the mouth/chin,
        # thumb close to lips, held steady close to the face throughout
        "id": "kula",
        "swahili_text": "Kula",
        "description": "A person holding the right hand in a loose closed fist with the thumb near the lips, keeping the fist steady close to the mouth in a repeated eating motion meaning to eat",
        "reference_img": None,
    },
    {
        # kunywa.mp4 — hand shaped like holding a cup, tipped up toward
        # the mouth, elbow rising as if drinking from a cup
        "id": "kunywa",
        "swahili_text": "Kunywa",
        "description": "A person forming the hand as if holding a small cup and tipping it upward toward the open mouth with the elbow rising, repeating a drinking motion meaning to drink",
        "reference_img": None,
    },
    {
        # kulala.mp4 — hand starts flat near the body then tilts and rests
        # against the tilted head/cheek, head leaning onto the flat hand
        # like a pillow
        "id": "kulala",
        "swahili_text": "Kulala",
        "description": "A person tilting the head to one side and resting the cheek against a flat open hand, as if using the hand as a pillow, meaning to sleep",
        "reference_img": None,
    },
    {
        # kusoma.mp4 — both hands held open together in front of the chest,
        # cupped like holding an open book, eyes looking down at the hands
        "id": "kusoma",
        "swahili_text": "Kusoma",
        "description": "A person holding both hands open and cupped together in front of the chest like holding an open book, with the head tilted down looking at the hands, meaning to read",
        "reference_img": None,
    },
    {
        # kuandika.mp4 — right hand pinched fingers making small repeated
        # writing motions across the open left palm, eyes looking down
        "id": "kuandika",
        "swahili_text": "Kuandika",
        "description": "A person pinching the right hand fingers together as if holding a pen and making small repeated side to side writing motions across the open left palm while looking down, meaning to write",
        "reference_img": None,
    },
    {
        # kutembea.mp4 — right hand fingers pointing down, making an
        # alternating stepping motion downward and forward like walking legs
        "id": "kutembea",
        "swahili_text": "Kutembea",
        "description": "A person holding the right hand with fingers pointing downward and moving it forward in a slow alternating stepping motion resembling walking legs, meaning to walk",
        "reference_img": None,
    },
    {
        # kukimbia.mp4 — both arms bent at elbows pumping forward and back
        # rapidly at the sides, body leaning slightly forward, fast motion
        "id": "kukimbia",
        "swahili_text": "Kukimbia",
        "description": "A person bending both arms at the elbows and pumping them quickly forward and backward at the sides of the body while leaning slightly forward, mimicking a fast running motion, meaning to run",
        "reference_img": None,
    },

    # ── Action verbs (S0017-S0024) ────────────────────────────────────────
    {
        # kufanya_kazi.mp4 — right flat hand chops down onto the open left
        # palm repeatedly, then both hands close into fists and make a
        # rhythmic pumping/working motion in front of the chest
        "id": "kufanya_kazi",
        "swahili_text": "Kufanya kazi",
        "description": "A person chopping the right flat hand down onto the open left palm, then closing both hands into fists and making a repeated rhythmic pumping motion in front of the chest, meaning to work",
        "reference_img": None,
    },
    {
        # usafiri.mp4 — right hand fingers pinched together near shoulder
        # height, arm extends and swings forward and to the side in a
        # smooth gliding motion, resembling a vehicle moving
        "id": "usafiri",
        "swahili_text": "Usafiri",
        "description": "A person holding the right hand with fingers pinched together near shoulder height then extending the arm forward and to the side in a smooth gliding motion resembling a moving vehicle, meaning transport or travel",
        "reference_img": None,
    },
    {
        # kucheza.mp4 — both hands start clasped at waist, then open and
        # move outward and down alternately at the sides in a loose
        # swinging, playful bouncing motion
        "id": "kucheza",
        "swahili_text": "Kucheza",
        "description": "A person starting with both hands clasped at the waist then opening them and swinging them outward and downward alternately at the sides in a loose playful bouncing motion, meaning to play",
        "reference_img": None,
    },
    {
        # kuimba.mp4 — right hand rises from chest to near the mouth with
        # fingers loosely open, opening and closing rhythmically in front
        # of the mouth as if the words are flowing out, mouth visibly
        # moving
        "id": "kuimba",
        "swahili_text": "Kuimba",
        "description": "A person raising the right hand from the chest to near the mouth with fingers loosely open, opening and closing the fingers rhythmically in front of the mouth as if words are flowing out while mouthing the words, meaning to sing",
        "reference_img": None,
    },
    {
        # kuona.mp4 — right index finger extends upward near the eye,
        # then the hand moves outward and forward away from the face,
        # finger leading the motion
        "id": "kuona",
        "swahili_text": "Kuona",
        "description": "A person raising the right index finger near the eye then moving the hand outward and forward away from the face with the finger leading the motion, meaning to see",
        "reference_img": None,
    },
    {
        # kusikia.mp4 — right index finger touches near the ear, then the
        # hand extends outward and forward, pointing away from the ear
        "id": "kusikia",
        "swahili_text": "Kusikia",
        "description": "A person touching the right index finger near the ear then extending the hand outward and forward, pointing away from the ear, meaning to hear",
        "reference_img": None,
    },
    {
        # kununua.mp4 — right hand starts low with fingers together
        # pointing down, then rotates and lifts as if placing something
        # into the open left palm, a giving/exchanging motion
        "id": "kununua",
        "swahili_text": "Kununua",
        "description": "A person starting with the right hand low and fingers together pointing down, then rotating and lifting the hand as if placing something into the open left palm in an exchanging motion, meaning to buy",
        "reference_img": None,
    },
    {
        # kuuza.mp4 — both hands held in front with fingers loosely open
        # and curled, palms up, making a small back-and-forth offering
        # motion as if presenting goods
        "id": "kuuza",
        "swahili_text": "Kuuza",
        "description": "A person holding both hands in front of the body with fingers loosely open and palms facing upward, making a small repeated back-and-forth offering motion as if presenting goods, meaning to sell",
        "reference_img": None,
    },

    # ── More action verbs (S0040-S0043) ───────────────────────────────────
    {
        # kupenda.mp4 — right flat hand crosses and presses against the
        # chest over the heart, held there briefly with a gentle circular
        # or pressing motion
        "id": "kupenda",
        "swahili_text": "Kupenda",
        "description": "A person crossing the right flat hand over the chest and pressing it against the heart, holding it there briefly with a gentle pressing motion, meaning to love",
        "reference_img": None,
    },
    {
        # kusaidia.mp4 — both hands come together at chest height, palms
        # up and cupped together as if lifting or offering support to
        # something resting on them
        "id": "kusaidia",
        "swahili_text": "Kusaidia",
        "description": "A person bringing both hands together at chest height with palms facing upward and cupped together, lifting them slightly as if offering support to something resting on them, meaning to help",
        "reference_img": None,
    },
    {
        # kujifunza.mp4 — right hand starts as a closed fist near the
        # shoulder, then fingers open and spread, then close again,
        # repeated opening and closing near the head, taking-in gesture
        "id": "kujifunza",
        "swahili_text": "Kujifunza",
        "description": "A person holding the right hand as a closed fist near the shoulder then repeatedly opening the fingers wide and closing them again near the head in a taking-in gesture, meaning to learn",
        "reference_img": None,
    },

    # ── Extended family (S0044-S0048) ─────────────────────────────────────
    {
        # dada.mp4 — right index finger raised, touching or pointing near
        # the cheek/jaw area, held steady with a slight downward stroke
        "id": "dada",
        "swahili_text": "Dada",
        "description": "A person raising the right index finger and touching it near the cheek or jawline with a slight downward stroke, held steady near the lower face, meaning sister",
        "reference_img": None,
    },
    {
        # kaka.mp4 — right index finger raised, touching or tapping near
        # the chin/mouth area repeatedly, similar to dada but positioned
        # closer to the chin/mouth
        "id": "kaka",
        "swahili_text": "Kaka",
        "description": "A person raising the right index finger and tapping it repeatedly near the chin or mouth area, meaning brother",
        "reference_img": None,
    },
    {
        # mjomba.mp4 — right hand starts near the jaw/cheek then the
        # index and middle fingers extend and touch near the temple or
        # side of the head, held briefly, hand lowers with a loose curl
        "id": "mjomba",
        "swahili_text": "Mjomba",
        "description": "A person starting with the right hand near the jaw then extending the index and middle fingers to touch near the temple or side of the head, holding briefly before lowering the hand, meaning uncle",
        "reference_img": None,
    },
    {
        # shangazi.mp4 — right flat hand starts near the chin/cheek then
        # sweeps down and across the chest in a smooth diagonal motion,
        # ending with the hand extended to the side, similar motion
        # pattern to mke but starting higher near the face
        "id": "shangazi",
        "swahili_text": "Shangazi",
        "description": "A person starting with the right flat hand near the chin or cheek then sweeping it down and across the chest in a smooth diagonal motion, ending with the hand extended to the side, meaning aunt",
        "reference_img": None,
    },
    {
        # bibi.mp4 — body leans forward with a slight bow, right hand
        # gesturing loosely near the waist/hip, bent posture suggesting
        # an elderly or respectful stance
        "id": "bibi",
        "swahili_text": "Bibi",
        "description": "A person leaning the body forward in a slight bow with the right hand gesturing loosely near the waist or hip, the bent posture suggesting an elderly figure, meaning grandmother",
        "reference_img": None,
    },

    # ── Family members (S0025-S0027, S0057-S0060) ────────────────────────
    {
        # mama.mp4 — right index finger raised near the chin/cheek,
        # held steady near the lower face
        "id": "mama",
        "swahili_text": "Mama",
        "description": "A person raising the right index finger and holding it steady near the chin or lower cheek, meaning mother",
        "reference_img": None,
    },
    {
        # baba.mp4 — right hand loosely closed near the chin, thumb and
        # fingers near the mouth/chin area, held steady close to the face
        "id": "baba",
        "swahili_text": "Baba",
        "description": "A person holding the right hand loosely closed near the chin with the thumb close to the mouth, keeping the hand steady near the lower face, meaning father",
        "reference_img": None,
    },
    {
        # familia.mp4 — both hands held together at chest height, fingers
        # interlaced or touching in a circular clasped shape, then one hand
        # lifts slightly with fingers spread before returning to clasp
        "id": "familia",
        "swahili_text": "Familia",
        "description": "A person holding both hands clasped together at chest height in a circular shape, briefly lifting one hand with fingers spread open before bringing the hands back together, meaning family",
        "reference_img": None,
    },
    {
        # S0057_Babu.mp4 — RE-CHECKED with full frame sequence
        # Right hand starts near chest/waist with fingers moving,
        # then touches/pats the top of the head twice,
        # then lowers into a loose fist near the hip while the body
        # bends slightly forward
        "id": "babu",
        "swahili_text": "Babu",
        "description": "A person touching the top of the head with the right hand and patting it, then lowering the hand into a loose fist near the hip while bending the body slightly forward, meaning grandfather",
        "reference_img": None,
    },
    {
        # S0058_Mke.mp4 — RE-CHECKED with full frame sequence
        # Right flat hand starts near the chin/mouth area then sweeps
        # downward and outward across the chest in one continuous
        # smooth diagonal motion, ending with the hand extended out
        # to the side at chest height
        "id": "mke",
        "swahili_text": "Mke",
        "description": "A person starting with the right flat hand near the chin then sweeping it downward and outward across the chest in one smooth diagonal motion, ending with the hand extended to the side at chest height, meaning wife",
        "reference_img": None,
    },
    {
        # S0059_Mume.mp4 — RE-CHECKED with full frame sequence
        # Right hand starts open at chest level, gesturing while speaking,
        # then the fingers bunch together and the hand rises to touch
        # or tap near the chin/mouth area, held briefly at the chin
        "id": "mume",
        "swahili_text": "Mume",
        "description": "A person starting with the right hand open at chest level then bunching the fingers together and raising the hand to tap or touch near the chin, holding it briefly at the chin, meaning husband",
        "reference_img": None,
    },
    {
        # S0060_Mtoto.mp4 — RE-CHECKED with full frame sequence
        # Right hand held at chest height with fingers loosely open,
        # rubbing or rolling the fingers and thumb together repeatedly
        # (a fidgeting/small motion), hand gradually lowers while
        # head tilts down to look at the hand
        "id": "mtoto",
        "swahili_text": "Mtoto",
        "description": "A person holding the right hand at chest height with fingers loosely open, rubbing the fingers and thumb together in a small repeated motion while the hand gradually lowers and the head tilts down to look at it, meaning child",
        "reference_img": None,
    },

    # ── Question words (S0028-S0031, S0033) ────────────────────────────
    {
        # vipi.mp4 — both hands raised with fingers curled loosely,
        # shaking or wiggling side to side at chest/shoulder height in a
        # questioning gesture
        "id": "vipi",
        "swahili_text": "Vipi",
        "description": "A person raising both hands with fingers loosely curled at chest or shoulder height and shaking or wiggling them side to side in a questioning gesture, meaning how",
        "reference_img": None,
    },
    {
        # kwanini.mp4 — right hand starts near the chin/temple then the
        # index finger extends and points forward and down, eyebrows
        # raised in a questioning expression
        "id": "kwanini",
        "swahili_text": "Kwanini",
        "description": "A person starting with the right hand near the chin or temple then extending the index finger to point forward and downward with a questioning facial expression, meaning why",
        "reference_img": None,
    },
    {
        # lini.mp4 — right hand starts in a fist near the body then the
        # index finger extends upward and circles or taps, ending with an
        # open smiling questioning expression
        "id": "lini",
        "swahili_text": "Lini",
        "description": "A person starting with the right hand in a closed fist then extending the index finger upward and making a small circling or tapping motion near shoulder height, meaning when",
        "reference_img": None,
    },
    {
        # wapi.mp4 — right hand held up with index, middle showing a
        # count-like shape (two or three fingers extended), palm facing
        # outward, held steady at chest height
        "id": "wapi",
        "swahili_text": "Wapi",
        "description": "A person holding up the right hand with two or three fingers extended and the palm facing outward at chest height, moving it slightly side to side in a searching questioning gesture, meaning where",
        "reference_img": None,
    },
    {
        # nini.mp4 — right hand starts with fingers together pointing up
        # near the body then extends outward with index finger pointing
        # forward, smiling questioning expression
        "id": "nini",
        "swahili_text": "Nini",
        "description": "A person starting with the right hand fingers together near the body then extending the arm outward with the index finger pointing forward in a questioning gesture, meaning what",
        "reference_img": None,
    },

    # ── Pronouns (S0034-S0039) ────────────────────────────────────────────
    {
        # wao.mp4 — right index finger extended, arm sweeping and pointing
        # outward to the side, away from the body, indicating a group
        "id": "wao",
        "swahili_text": "Wao",
        "description": "A person extending the right index finger and sweeping the arm outward to the side away from the body, pointing toward a distant group, meaning they or them",
        "reference_img": None,
    },
    {
        # nyinyi.mp4 — right hand open, palm up, sweeping from one side to
        # the other in front of the body at waist height, indicating a
        # group facing the signer
        "id": "nyinyi",
        "swahili_text": "Nyinyi",
        "description": "A person holding the right hand open with the palm facing upward and sweeping it from one side to the other in front of the body at waist height, indicating a group of people, meaning you all",
        "reference_img": None,
    },
    {
        # sisi.mp4 — right index finger pointing and moving back and forth
        # between the signer's own chest and outward, alternating motion
        "id": "sisi",
        "swahili_text": "Sisi",
        "description": "A person pointing the right index finger and moving it back and forth between their own chest and an outward direction in an alternating motion, meaning we or us",
        "reference_img": None,
    },
    {
        # yeye.mp4 — right index finger extended, pointing off to the side
        # away from the signer's own body, arm slightly bent
        "id": "yeye",
        "swahili_text": "Yeye",
        "description": "A person extending the right index finger and pointing it off to the side away from their own body with the arm slightly bent, meaning he or she",
        "reference_img": None,
    },
    {
        # wewe.mp4 — right index finger extended, pointing directly forward
        # toward the viewer/other person, held steady
        "id": "wewe",
        "swahili_text": "Wewe",
        "description": "A person extending the right index finger and pointing it directly forward toward the other person, held steady, meaning you",
        "reference_img": None,
    },
    {
        # mimi.mp4 — right hand in a loose fist or thumb pointing, tapping
        # or pressing against the signer's own chest
        "id": "mimi",
        "swahili_text": "Mimi",
        "description": "A person holding the right hand in a loose fist with the thumb pointing inward and tapping or pressing it against their own chest, meaning I or me",
        "reference_img": None,
    },

    # ── Professions / roles (S0061-S0070) ────────────────────────────────
    {
        # rafiki.mp4 — both hands come together with index fingers hooking
        # or interlocking in front of the chest, a linking gesture
        "id": "rafiki",
        "swahili_text": "Rafiki",
        "description": "A person hooking or interlocking the index fingers of both hands together in front of the chest in a linking gesture, meaning friend",
        "reference_img": None,
    },
    {
        # mwanafunzi.mp4 — right hand fingers together pointing to the
        # temple/forehead then both hands come down and open at chest
        # height like holding a book, referencing learning
        "id": "mwanafunzi",
        "swahili_text": "Mwanafunzi",
        "description": "A person touching the fingers of the right hand near the temple or forehead then bringing both hands down to an open position at chest height as if holding a book, meaning student",
        "reference_img": None,
    },
    {
        # mwalimu.mp4 — right hand index finger raised near the temple,
        # then the hand opens and moves outward and down as if
        # explaining or presenting knowledge to others
        "id": "mwalimu",
        "swahili_text": "Mwalimu",
        "description": "A person raising the right index finger near the temple then opening the hand and moving it outward and downward in an explaining or presenting motion, meaning teacher",
        "reference_img": None,
    },
    {
        # daktari.mp4 — right hand fingers pinched together tapping on the
        # inside of the opposite wrist or forearm, referencing checking a
        # pulse or medical examination
        "id": "daktari",
        "swahili_text": "Daktari",
        "description": "A person pinching the fingers of the right hand together and tapping them on the inside of the opposite wrist or forearm as if checking a pulse, meaning doctor",
        "reference_img": None,
    },
    {
        # muuguzi.mp4 — right hand raised above the head with fingers bent
        # into a claw or cross shape, held up and slightly rotated, then
        # lowers near the shoulder
        "id": "muuguzi",
        "swahili_text": "Muuguzi",
        "description": "A person raising the right hand above the head with fingers bent into a claw or cross shape, holding it up and slightly rotating before lowering it near the shoulder, meaning nurse",
        "reference_img": None,
    },
    {
        # dereva.mp4 — both hands raised in front of the body forming
        # fists as if gripping a steering wheel, making a turning motion
        "id": "dereva",
        "swahili_text": "Dereva",
        "description": "A person raising both hands in front of the body with closed fists as if gripping a steering wheel and making a turning motion side to side, meaning driver",
        "reference_img": None,
    },
    {
        # polisi.mp4 — right hand raised in a flat salute position near the
        # forehead/temple, held briefly then lowered
        "id": "polisi",
        "swahili_text": "Polisi",
        "description": "A person raising the right flat hand to the forehead or temple in a salute-like position, holding it briefly before lowering the arm, meaning police",
        "reference_img": None,
    },
    {
        # S0068_Mkuu.mp4 — RE-CHECKED with full frame sequence
        # Starts neutral, then right hand lowers and opens beside the hip,
        # then both hands rise together to shoulder height with index
        # fingers pointing straight up side by side, then both hands
        # open and lower back down to waist level with palms facing
        # slightly outward
        "id": "mkuu",
        "swahili_text": "Mkuu",
        "description": "A person raising both hands to shoulder height with the index fingers pointing straight up side by side, then opening both hands and lowering them back down to waist level with palms facing slightly outward, meaning boss or leader",
        "reference_img": None,
    },
    {
        # mfanyakazi.mp4 — right flat hand chops or taps against the open
        # left palm repeatedly, a working or hammering motion
        "id": "mfanyakazi",
        "swahili_text": "Mfanyakazi",
        "description": "A person tapping or chopping the edge of the right flat hand against the open left palm in a repeated working or hammering motion, meaning worker",
        "reference_img": None,
    },
    {
        # mgeni.mp4 — right hand starts near the body then sweeps outward
        # and downward away from the signer, an introducing or arriving
        # gesture
        "id": "mgeni",
        "swahili_text": "Mgeni",
        "description": "A person starting with the right hand near the body then sweeping it outward and downward away from themselves in an introducing or arriving gesture, meaning guest or visitor",
        "reference_img": None,
    },
]

print(f"✅ VOCABULARY loaded: {len(VOCABULARY)} signs")
for s in VOCABULARY:
    print(f"   {s['id']:<24} → {s['swahili_text']}")

✅ VOCABULARY loaded: 60 signs
   habari                   → Habari
   nzuri                    → Nzuri
   asante                   → Asante
   mbaya                    → Mbaya
   maji                     → Maji
   tafadhali_au_samahani    → Tafadhali au Samahani
   chakula                  → Chakula
   jina_langu               → Jina langu
   kwaheri                  → Kwaheri
   kula                     → Kula
   kunywa                   → Kunywa
   kulala                   → Kulala
   kusoma                   → Kusoma
   kuandika                 → Kuandika
   kutembea                 → Kutembea
   kukimbia                 → Kukimbia
   kufanya_kazi             → Kufanya kazi
   usafiri                  → Usafiri
   kucheza                  → Kucheza
   kuimba                   → Kuimba
   kuona                    → Kuona
   kusikia                  → Kusikia
   kununua                  → Kununua
   kuuza                    → Kuuza
   kupenda                  → Kupenda
   kusaidia    

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CELL 5 — Build text prototype embeddings (few-shot setup)

### For each sign we create a CLIP text embedding from its
### English description. When you have real images, this cell
### also averages in visual embeddings for stronger prototypes.

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

# ════════════════════════════════════════════════════════════════
# VIDEO_MAP  —  points to locally uploaded videos
# After uploading your 9 mp4 files via files.upload() they land
# at /content/<filename>.  Adjust names if yours differ.
# ════════════════════════════════════════════════════════════════
VIDEO_MAP = {
    "habari":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0001_Habari.mp4",
    "nzuri":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0002_Nzuri.mp4",
    "asante":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0003_Asante.mp4",
    "mbaya":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0004_Mbaya.mp4",
    "maji":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0005_Maji.mp4",
    "tafadhali_au_samahani": "/content/drive/MyDrive/ZanAI/nena project/dataset/S0006_Tafadhali_au_Samahani.mp4",
    "chakula":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0007_Chakula.mp4",
    "jina_langu":            "/content/drive/MyDrive/ZanAI/nena project/dataset/S0008_Jina_langu.mp4",
    "kwaheri":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0009_Kwaheri.mp4",
    "kula":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0010_kula.mp4",
    "kunywa":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0011_kunywa.mp4",
    "kulala":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0012_kulala.mp4",
    "kusoma":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0013_kusoma.mp4",
    "kuandika":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0014_kuandika.mp4",
    "kutembea":              "/content/drive/MyDrive/ZanAI/nena project/dataset/S0015_kutembea.mp4",
    "kukimbia":              "/content/drive/MyDrive/ZanAI/nena project/dataset/S0016_kukimbia.mp4",
    "kufanya_kazi":           "/content/drive/MyDrive/ZanAI/nena project/dataset/S0017_kufanya_kazi.mp4",
    "usafiri":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0018_usafiri.mp4",
    "kucheza":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0019_kucheza.mp4",
    "kuimba":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0020_kuimba.mp4",
    "kuona":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0021_kuona.mp4",
    "kusikia":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0022_kusikia.mp4",
    "kununua":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0023_kununua.mp4",
    "kuuza":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0024_kuuza.mp4",
    "kupenda":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0040_kupenda.mp4",
    "kusaidia":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0042_kusaidia.mp4",
    "kujifunza":              "/content/drive/MyDrive/ZanAI/nena project/dataset/S0043_kujifunza.mp4",
    "dada":                   "/content/drive/MyDrive/ZanAI/nena project/dataset/S0044_dada.mp4",
    "kaka":                   "/content/drive/MyDrive/ZanAI/nena project/dataset/S0045_kaka.mp4",
    "mjomba":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0046_mjomba.mp4",
    "shangazi":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0047_shangazi.mp4",
    "bibi":                   "/content/drive/MyDrive/ZanAI/nena project/dataset/S0048_bibi.mp4",
    "mama":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0025_Mama.mp4",
    "baba":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0026_Baba.mp4",
    "familia":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0027_Familia.mp4",
    "vipi":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0028_Vipi.mp4",
    "kwanini":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0029_Kwanini.mp4",
    "lini":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0030_Lini.mp4",
    "wapi":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0031_Wapi.mp4",
    "nini":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0033_Nini.mp4",
    "wao":                   "/content/drive/MyDrive/ZanAI/nena project/dataset/S0034_Wao.mp4",
    "nyinyi":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0035_Nyinyi.mp4",
    "sisi":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0036_Sisi.mp4",
    "yeye":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0037_Yeye.mp4",
    "wewe":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0038_Wewe.mp4",
    "mimi":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0039_Mimi.mp4",
    "babu":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0057_Babu.mp4",
    "mke":                   "/content/drive/MyDrive/ZanAI/nena project/dataset/S0058_Mke.mp4",
    "mume":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0059_Mume.mp4",
    "mtoto":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0060_Mtoto.mp4",
    "rafiki":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0061_Rafiki.mp4",
    "mwanafunzi":            "/content/drive/MyDrive/ZanAI/nena project/dataset/S0062_Mwanafunzi.mp4",
    "mwalimu":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0063_Mwalimu.mp4",
    "daktari":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0064_Daktari.mp4",
    "muuguzi":               "/content/drive/MyDrive/ZanAI/nena project/dataset/S0065_Muuguzi.mp4",
    "dereva":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0066_Dereva.mp4",
    "polisi":                "/content/drive/MyDrive/ZanAI/nena project/dataset/S0067_Polisi.mp4",
    "mkuu":                  "/content/drive/MyDrive/ZanAI/nena project/dataset/S0068_Mkuu.mp4",
    "mfanyakazi":            "/content/drive/MyDrive/ZanAI/nena project/dataset/S0069_Mfanyakazi.mp4",
    "mgeni":                 "/content/drive/MyDrive/ZanAI/nena project/dataset/S0070_Mgeni.mp4",
}

print(f"✅ VIDEO_MAP loaded: {len(VIDEO_MAP)} sign videos")
for sid in VIDEO_MAP:
    print(f"   {sid}")

FRAME_DIR = Path("/content/nena_frames")

def extract_frames_from_videos(video_map, out_dir, fps_extract=3.0, max_frames=15):
    out_dir.mkdir(parents=True, exist_ok=True)
    frame_paths = {}
    for sign_id, video_path in video_map.items():
        sign_dir = out_dir / sign_id
        sign_dir.mkdir(exist_ok=True)
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"  ⚠  Cannot open {video_path} — skipping {sign_id}")
            frame_paths[sign_id] = []
            continue
        video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step  = max(1, int(video_fps / fps_extract))
        saved = []
        frame_idx = 0
        while len(saved) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % step == 0:
                fpath = sign_dir / f"frame_{len(saved):04d}.jpg"
                cv2.imwrite(str(fpath), frame)
                saved.append(str(fpath))
            frame_idx += 1
        cap.release()
        frame_paths[sign_id] = saved
        print(f"  ✅ {sign_id:<16} → {len(saved)} frames extracted")
    return frame_paths

print("Extracting frames from TSL videos…")
FRAME_PATHS = extract_frames_from_videos(VIDEO_MAP, FRAME_DIR)
print(f"\nDone. Frames saved to {FRAME_DIR}")

# Attach frame paths to VOCABULARY
for sign in VOCABULARY:
    paths = FRAME_PATHS.get(sign["id"], [])
    sign["reference_img"] = paths

print("\nVocabulary updated:")
for sign in VOCABULARY:
    n = len(sign["reference_img"])
    print(f"  {sign['id']:<16} {n} frames")

def build_prototypes(vocabulary, model, tokenizer, preprocess, device,
                     image_weight=3.0, min_frames_for_image_weight=3):
    prototypes = {}
    with torch.no_grad():
        for sign in vocabulary:
            prompts = [
                sign["description"],
                f"A sign language gesture: {sign['description']}",
                f"Tanzanian sign language sign meaning {sign['swahili_text']}: {sign['description']}",
            ]
            tokens     = tokenizer(prompts).to(device)
            text_feats = model.encode_text(tokens)
            text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
            text_proto = text_feats.mean(dim=0, keepdim=True)
            text_proto = text_proto / text_proto.norm(dim=-1, keepdim=True)

            img_paths   = sign.get("reference_img") or []
            valid_paths = [p for p in img_paths if Path(p).exists()]

            if len(valid_paths) >= min_frames_for_image_weight:
                img_feats_list = []
                for p in valid_paths:
                    try:
                        img  = preprocess(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
                        feat = model.encode_image(img)
                        feat = feat / feat.norm(dim=-1, keepdim=True)
                        img_feats_list.append(feat)
                    except Exception as e:
                        print(f"    ⚠  Skipping frame {p}: {e}")
                if img_feats_list:
                    img_proto     = torch.cat(img_feats_list, dim=0).mean(dim=0, keepdim=True)
                    img_proto     = img_proto / img_proto.norm(dim=-1, keepdim=True)
                    weight_copies = [img_proto] * int(image_weight) + [text_proto]
                    proto         = torch.cat(weight_copies, dim=0).mean(dim=0, keepdim=True)
                    proto         = proto / proto.norm(dim=-1, keepdim=True)
                    src = f"image×{int(image_weight)} + text×1  ({len(img_feats_list)} frames)"
                else:
                    proto = text_proto
                    src   = "text-only (image load failed)"
            else:
                proto = text_proto
                src   = f"text-only (only {len(valid_paths)} frames, need ≥{min_frames_for_image_weight})"

            prototypes[sign["id"]] = proto
            print(f"  {sign['id']:<16} [{src}]")

    print(f"\n✅ Built {len(prototypes)} prototype embeddings.")
    print(f"   Embedding dim : {list(prototypes.values())[0].shape[-1]}")
    print(f"   Device        : {list(prototypes.values())[0].device}")
    return prototypes

PROTOTYPES = build_prototypes(VOCABULARY, model, tokenizer, preprocess, device)


✅ VIDEO_MAP loaded: 60 sign videos
   habari
   nzuri
   asante
   mbaya
   maji
   tafadhali_au_samahani
   chakula
   jina_langu
   kwaheri
   kula
   kunywa
   kulala
   kusoma
   kuandika
   kutembea
   kukimbia
   kufanya_kazi
   usafiri
   kucheza
   kuimba
   kuona
   kusikia
   kununua
   kuuza
   kupenda
   kusaidia
   kujifunza
   dada
   kaka
   mjomba
   shangazi
   bibi
   mama
   baba
   familia
   vipi
   kwanini
   lini
   wapi
   nini
   wao
   nyinyi
   sisi
   yeye
   wewe
   mimi
   babu
   mke
   mume
   mtoto
   rafiki
   mwanafunzi
   mwalimu
   daktari
   muuguzi
   dereva
   polisi
   mkuu
   mfanyakazi
   mgeni
Extracting frames from TSL videos…
  ✅ habari           → 6 frames extracted
  ✅ nzuri            → 5 frames extracted
  ✅ asante           → 6 frames extracted
  ✅ mbaya            → 4 frames extracted
  ✅ maji             → 6 frames extracted
  ✅ tafadhali_au_samahani → 8 frames extracted
  ✅ chakula          → 5 frames extracted
  ✅ jina_langu       

# CELL 6 — Few-shot sign classifier

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Few-shot sign classifier (corrected + improved)
# ════════════════════════════════════════════════════════════════════════

# ── Imports ──────────────────────────────────────────────────────────────
import torch
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from typing import Optional          # needed for Optional[str] type hint

# ── Vocabulary lookup (convenience dict) ────────────────────────────────
VOCAB_LOOKUP = {sign["id"]: sign["swahili_text"] for sign in VOCABULARY}


# ════════════════════════════════════════════════════════════════════════
# CORE: embed a single frame
# ════════════════════════════════════════════════════════════════════════

def embed_frame(frame_bgr: np.ndarray, model, preprocess, device: str) -> torch.Tensor:
    """
    Takes an OpenCV BGR frame, returns a normalised CLIP image embedding (1, D).
    """
    rgb     = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    tensor  = preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat                                          # (1, D)


# ════════════════════════════════════════════════════════════════════════
# SINGLE-FRAME classifier (unchanged logic, threshold raised to 0.28)
# ════════════════════════════════════════════════════════════════════════

def classify_sign(
    frame_bgr: np.ndarray,
    prototypes: dict,
    model,
    preprocess,
    device: str,
    threshold: float = 0.28,        # raised from 0.20 — reduces false positives
) -> tuple[Optional[str], float]:
    """
    Classifies a single video frame against all prototype embeddings.

    Returns:
        (sign_id, confidence) if match found above threshold, else (None, score).

    Note: fine for real-time / per-frame use, but noisy on individual frames.
          Use classify_sign_from_clip() when you have a short clip available.
    """
    query = embed_frame(frame_bgr, model, preprocess, device)   # (1, D)

    best_id, best_score = None, -1.0
    for sign_id, proto in prototypes.items():
        score = (query @ proto.T).item()    # cosine similarity ∈ [−1, 1]
        if score > best_score:
            best_score = score
            best_id    = sign_id

    if best_score < threshold:
        return None, best_score

    return best_id, best_score


# ════════════════════════════════════════════════════════════════════════
# CLIP-AVERAGED classifier (new) — recommended for offline / clip-level use
#
# Why this is better:
#   build_prototypes() averages frame embeddings on the REFERENCE side.
#   This function does the same on the QUERY side, so the comparison is
#   symmetric: avg_frames ↔ avg_frames, not single_frame ↔ avg_frames.
#   Averaging over even 3–5 frames filters out motion blur and pose noise,
#   typically lifting cosine similarity by 0.04–0.08 on correct matches.
# ════════════════════════════════════════════════════════════════════════

def classify_sign_from_clip(
    frames: list,                   # list of BGR np.ndarray frames
    prototypes: dict,
    model,
    preprocess,
    device: str,
    threshold: float = 0.28,
) -> tuple[Optional[str], float]:
    """
    Classifies a short video clip (list of frames) by averaging embeddings
    before comparing against prototypes.

    Returns:
        (sign_id, confidence) if match found above threshold, else (None, score).

    Usage:
        frames = [frame1, frame2, ..., frame_n]   # BGR, from cv2
        sign_id, conf = classify_sign_from_clip(frames, PROTOTYPES, model, preprocess, device)
    """
    if not frames:
        return None, 0.0

    # Embed every frame then average → one stable query vector
    feats = [embed_frame(f, model, preprocess, device) for f in frames]
    query = torch.cat(feats, dim=0).mean(dim=0, keepdim=True)   # (1, D)
    query = query / query.norm(dim=-1, keepdim=True)             # re-normalise

    best_id, best_score = None, -1.0
    for sign_id, proto in prototypes.items():
        score = (query @ proto.T).item()
        if score > best_score:
            best_score = score
            best_id    = sign_id

    return (best_id, best_score) if best_score >= threshold else (None, best_score)


# ════════════════════════════════════════════════════════════════════════
# QUICK SANITY CHECK
# Run this after building PROTOTYPES to make sure the classifier works.
# ════════════════════════════════════════════════════════════════════════

def sanity_check_classifier(
    video_map: dict,
    prototypes: dict,
    model,
    preprocess,
    device: str,
    n_frames: int = 5,              # how many frames to sample per video
):
    """
    Opens each source video, grabs n_frames evenly spaced frames,
    runs classify_sign_from_clip(), and prints whether it predicts
    the correct sign label.

    Useful to catch threshold or weighting bugs early before
    integrating with live camera input.
    """
    print("── Sanity check ──────────────────────────────────────────")
    correct, total = 0, 0

    for sign_id, video_path in video_map.items():
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"  ⚠  Cannot open {video_path}")
            continue

        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        indices = np.linspace(0, n_total - 1, n_frames, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frames.append(frame)
        cap.release()

        pred_id, conf = classify_sign_from_clip(
            frames, prototypes, model, preprocess, device
        )

        ok     = pred_id == sign_id
        symbol = "✅" if ok else "❌"
        pred_label = VOCAB_LOOKUP.get(pred_id, "—") if pred_id else "below threshold"
        true_label = VOCAB_LOOKUP[sign_id]
        print(f"  {symbol} {sign_id:<16}  true={true_label:<20} pred={pred_label}  (conf={conf:.3f})")

        correct += int(ok)
        total   += 1

    print(f"\n  Accuracy: {correct}/{total} = {correct/total*100:.0f}%")
    print("──────────────────────────────────────────────────────────")


# Run the sanity check against your 9 source videos
sanity_check_classifier(VIDEO_MAP, PROTOTYPES, model, preprocess, device)

── Sanity check ──────────────────────────────────────────
  ✅ habari            true=Habari               pred=Habari  (conf=0.958)
  ✅ nzuri             true=Nzuri                pred=Nzuri  (conf=0.956)
  ✅ asante            true=Asante               pred=Asante  (conf=0.958)
  ✅ mbaya             true=Mbaya                pred=Mbaya  (conf=0.961)
  ✅ maji              true=Maji                 pred=Maji  (conf=0.956)
  ✅ tafadhali_au_samahani  true=Tafadhali au Samahani pred=Tafadhali au Samahani  (conf=0.956)
  ✅ chakula           true=Chakula              pred=Chakula  (conf=0.953)
  ✅ jina_langu        true=Jina langu           pred=Jina langu  (conf=0.956)
  ✅ kwaheri           true=Kwaheri              pred=Kwaheri  (conf=0.956)
  ✅ kula              true=Kula                 pred=Kula  (conf=0.958)
  ✅ kunywa            true=Kunywa               pred=Kunywa  (conf=0.954)
  ✅ kulala            true=Kulala               pred=Kulala  (conf=0.957)
  ✅ kusoma            true=Kusom

# 7 — TTS Voice Router

### Implements the "Profile Mode" from the architecture document:
###   Profile Alpha → Male Swahili voice (sw-TZ-Standard-B)
###   Profile Beta  → Female Swahili voice (sw-TZ-Standard-A)
### Two backends available:
###   Backend A: Google Cloud TTS (needs GOOGLE_APPLICATION_CREDENTIALS)
###   Backend B: gTTS (free, no API key, Swahili supported)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL — TTS Voice Router  (Nena Project)
#
# Implements Profile Mode from the architecture document:
#   Profile Alpha → Male   Swahili voice (sw-TZ-Standard-B)
#   Profile Beta  → Female Swahili voice (sw-TZ-Standard-A)
#
# Two backends:
#   "gtts" → free, no API key, no gender switching (prototype)
#   "gcp"  → Google Cloud TTS, gender-correct sw-TZ voices (production)
# ════════════════════════════════════════════════════════════════════════

# ── Imports ──────────────────────────────────────────────────────────────
import os
import tempfile
from gtts import gTTS
from IPython.display import Audio, display


# ════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

# Voice profiles — mirrors the app settings layer
# Change "voice" to "Male" for Profile Alpha, "Female" for Profile Beta
USER_PROFILE = {
    "voice":    "Female",   # "Male" = Profile Alpha | "Female" = Profile Beta
    "speed":    1.0,        # 0.25–4.0 for GCP; gTTS ignores this
    "language": "sw",       # Swahili BCP-47
}

# Maps profile setting → Google Cloud TTS voice name
# Standard-A = Female (Profile Beta), Standard-B = Male (Profile Alpha)
VOICE_MAP = {
    "Male":   "sw-TZ-Standard-B",   # Profile Alpha
    "Female": "sw-TZ-Standard-A",   # Profile Beta
}

# GCP service account key path — set once here, used by text_to_swahili_speech_gcp()
# Upload your JSON key to Colab then update this path:
GCP_KEY_PATH = "/content/drive/MyDrive/ZanAI/nena project/gcp_key.json"


# ════════════════════════════════════════════════════════════════════════
# BACKEND 1 — gTTS (free, prototype)
#
# Uses Google Translate TTS engine under the hood — different engine from
# Cloud TTS, so audio quality will differ from the GCP backend.
# Swahili is supported but gender switching is NOT available.
# Both Male/Female profiles will produce the same voice here.
# ════════════════════════════════════════════════════════════════════════

def text_to_swahili_speech_gtts(
    text: str,
    voice_setting: str,
    speed: float = 1.0,             # accepted for API consistency; gTTS ignores it
    output_path: str = None,
) -> str:
    """
    Synthesises Swahili speech using gTTS (free, no API key).
    Gender switching is not supported — both profiles produce the same voice.
    Use the GCP backend for gender-correct sw-TZ voices.

    Returns:
        Path to the saved MP3 file.
    """
    if output_path is None:
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        output_path = tmp.name

    tts = gTTS(text=text, lang="sw", slow=False)
    tts.save(output_path)

    voice_id = VOICE_MAP.get(voice_setting, "sw-TZ-Standard-A")
    print(f"[Nena | gTTS  ] Text    : '{text}'")
    print(f"[Nena | gTTS  ] Profile : {voice_setting} ({voice_id}) — gender ignored by gTTS")
    print(f"[Nena | gTTS  ] Audio   → {output_path}")
    return output_path


# ════════════════════════════════════════════════════════════════════════
# BACKEND 2 — Google Cloud TTS (production, gender-correct)
#
# Setup:
#   1. Create a GCP project, enable Cloud Text-to-Speech API.
#   2. Download service-account JSON → upload to Colab (or Google Drive).
#   3. Update GCP_KEY_PATH above, or set:
#        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/key.json"
# ════════════════════════════════════════════════════════════════════════

def text_to_swahili_speech_gcp(
    text: str,
    voice_setting: str,
    speed: float = 1.0,
    output_path: str = None,
) -> str:
    """
    Synthesises Swahili speech using Google Cloud Neural TTS.
    Supports gender-correct sw-TZ voices and speaking rate control.

    Returns:
        Path to the saved MP3 file.

    Raises:
        RuntimeError if GCP credentials are not configured.
        ImportError if google-cloud-texttospeech is not installed.
    """
    # ── Credential check ────────────────────────────────────────────────
    # Set from GCP_KEY_PATH if not already in environment
    if not os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
        if os.path.exists(GCP_KEY_PATH):
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_KEY_PATH
            print(f"[Nena | GCP   ] Credentials loaded from {GCP_KEY_PATH}")
        else:
            raise RuntimeError(
                "[Nena | GCP] No credentials found.\n"
                f"  Expected key at: {GCP_KEY_PATH}\n"
                "  Either upload your GCP service-account JSON to that path,\n"
                "  or set os.environ['GOOGLE_APPLICATION_CREDENTIALS'] manually."
            )

    # ── Import (deferred so missing package only errors when GCP is used) ──
    try:
        from google.cloud import texttospeech
    except ImportError:
        raise ImportError(
            "[Nena | GCP] google-cloud-texttospeech is not installed.\n"
            "  Run:  !pip install google-cloud-texttospeech"
        )

    # ── Synthesis ────────────────────────────────────────────────────────
    client           = texttospeech.TextToSpeechClient()
    synthesis_input  = texttospeech.SynthesisInput(text=text)

    voice_id = VOICE_MAP.get(voice_setting, "sw-TZ-Standard-A")
    gender   = (
        texttospeech.SsmlVoiceGender.MALE
        if voice_setting == "Male"
        else texttospeech.SsmlVoiceGender.FEMALE
    )

    voice = texttospeech.VoiceSelectionParams(
        language_code="sw-TZ",
        name=voice_id,
        ssml_gender=gender,
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3,
        speaking_rate=speed,            # uses passed value, not global
    )

    response = client.synthesize_speech(
        input=synthesis_input,
        voice=voice,
        audio_config=audio_config,
    )

    if output_path is None:
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        output_path = tmp.name

    with open(output_path, "wb") as f:
        f.write(response.audio_content)

    print(f"[Nena | GCP   ] Text    : '{text}'")
    print(f"[Nena | GCP   ] Profile : {voice_setting} → {voice_id}  speed={speed}")
    print(f"[Nena | GCP   ] Audio   → {output_path}")
    return output_path


# ════════════════════════════════════════════════════════════════════════
# MAIN ENTRY POINTS
# ════════════════════════════════════════════════════════════════════════

def speak(
    text: str,
    profile: dict = None,           # None defaults to USER_PROFILE at call time
    backend: str = "gtts",          # "gtts" or "gcp"
    output_path: str = None,
) -> str:
    """
    Synthesises text to Swahili speech and saves to an MP3 file.

    Args:
        text        : Swahili text to synthesise.
        profile     : Voice profile dict (defaults to USER_PROFILE).
                      Keys: "voice" ("Male"/"Female"), "speed" (float).
        backend     : "gtts" (free, no gender) | "gcp" (production, gender-correct).
        output_path : Optional path for the output MP3. Temp file if None.

    Returns:
        Path to the saved MP3 file.

    Example:
        speak("Habari", backend="gtts")
        speak("Habari", profile={"voice": "Male", "speed": 1.2}, backend="gcp")
    """
    # Read profile at call time (avoids mutable-default-argument pitfall)
    if profile is None:
        profile = USER_PROFILE

    voice_setting = profile.get("voice", "Female")
    speed         = profile.get("speed", 1.0)

    if backend == "gcp":
        return text_to_swahili_speech_gcp(text, voice_setting, speed=speed, output_path=output_path)
    return text_to_swahili_speech_gtts(text, voice_setting, speed=speed, output_path=output_path)


def speak_and_play(
    text: str,
    profile: dict = None,
    backend: str = "gtts",
) -> str:
    """
    Synthesises text, saves to MP3, and plays it inline in Colab.

    Returns:
        Path to the saved MP3 file.

    Example:
        speak_and_play("Asante sana")
        speak_and_play("Habari", profile={"voice": "Male", "speed": 1.0}, backend="gcp")
    """
    path = speak(text, profile=profile, backend=backend)
    display(Audio(path, autoplay=True))
    return path


# ════════════════════════════════════════════════════════════════════════
# SANITY CHECK
# Runs through all 9 vocabulary signs, speaks each one, confirms no errors.
# ════════════════════════════════════════════════════════════════════════

def tts_sanity_check(backend: str = "gtts"):
    """
    Speaks each sign's Swahili label through the selected backend.
    Plays audio for the last sign only to keep the check quick.
    Prints a pass/fail line for each sign.
    """
    print(f"── TTS sanity check ({backend}) ──────────────────────────────")
    passed, failed = 0, []

    for sign in VOCABULARY:
        try:
            path = speak(sign["swahili_text"], backend=backend)
            print(f"  ✅ {sign['id']:<16} → '{sign['swahili_text']}'  ({path})")
            passed += 1
        except Exception as e:
            print(f"  ❌ {sign['id']:<16} → ERROR: {e}")
            failed.append(sign["id"])

    print(f"\n  Passed: {passed}/{len(VOCABULARY)}")
    if failed:
        print(f"  Failed: {failed}")

    # Play the last successfully generated file so you can hear it
    if passed > 0:
        last_path = speak(VOCABULARY[-1]["swahili_text"], backend=backend)
        print(f"\n  Playing last sign: '{VOCABULARY[-1]['swahili_text']}'")
        display(Audio(last_path, autoplay=True))

    print("──────────────────────────────────────────────────────────")


# Run the sanity check (change to "gcp" once your key is configured)
tts_sanity_check(backend="gtts")

── TTS sanity check (gtts) ──────────────────────────────
[Nena | gTTS  ] Text    : 'Habari'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmp_6blhmmb.mp3
  ✅ habari           → 'Habari'  (/tmp/tmp_6blhmmb.mp3)
[Nena | gTTS  ] Text    : 'Nzuri'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpahias885.mp3
  ✅ nzuri            → 'Nzuri'  (/tmp/tmpahias885.mp3)
[Nena | gTTS  ] Text    : 'Asante'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpef_f5w6a.mp3
  ✅ asante           → 'Asante'  (/tmp/tmpef_f5w6a.mp3)
[Nena | gTTS  ] Text    : 'Mbaya'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpnpgq39f8.mp3
  ✅ mbaya            → 'Mbaya'  (/tmp/tmpnpgq39f8.mp3)
[Nena | gTTS  ] Text    : 'Maji'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) —

──────────────────────────────────────────────────────────


# 8 — End-to-end demo (single image or video frame)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — End-to-end demo  (updated: uses clip-level classifier)
# ════════════════════════════════════════════════════════════════
from IPython.display import Audio, display
import cv2, numpy as np
from typing import Optional

def run_demo(image_path=None, frame_bgr=None, video_path=None, n_frames=8):
    """
    Three modes:
      image_path  → classify a single saved JPG/PNG
      frame_bgr   → classify a raw OpenCV frame
      video_path  → sample n_frames from a video clip (most accurate)
    If nothing is passed, uses a random frame from the first sign's extracted frames.
    """
    print("─" * 52)
    print("NENA · Tanzanian Sign Language · Inference Demo")
    print("─" * 52)

    # ── Collect frames ─────────────────────────────────────────
    frames = []

    if video_path:
        cap     = cv2.VideoCapture(video_path)
        total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        indices = np.linspace(0, total - 1, n_frames, dtype=int)
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, f = cap.read()
            if ret:
                frames.append(f)
        cap.release()

    elif image_path:
        f = cv2.imread(image_path)
        if f is None:
            print(f"❌ Could not load: {image_path}")
            return
        frames = [f]

    elif frame_bgr is not None:
        frames = [frame_bgr]

    else:
        # Use a real extracted frame from FRAME_DIR as default demo
        first_sign = VOCABULARY[0]["id"]
        demo_frames = list((FRAME_DIR / first_sign).glob("*.jpg"))
        if demo_frames:
            frames = [cv2.imread(str(p)) for p in demo_frames[:8]]
            print(f"ℹ️  Demo: using extracted frames from '{first_sign}'")
        else:
            frames = [np.random.randint(0, 255, (312, 640, 3), dtype=np.uint8)]
            print("ℹ️  Demo: using synthetic noise frame (no real frames found)")

    # ── Classify ───────────────────────────────────────────────
    sign_id, confidence = classify_sign_from_clip(
        frames, PROTOTYPES, model, preprocess, device
    )

    if sign_id is None:
        print(f"[Nena] No sign recognised (best cosine: {confidence:.3f})")
        print("       Try lower threshold or add more reference frames.")
        return

    swahili_text = VOCAB_LOOKUP[sign_id]
    print(f"[Nena] Sign detected : {sign_id}")
    print(f"[Nena] Swahili text  : {swahili_text}")
    print(f"[Nena] Confidence    : {confidence:.3f}")

    # Show top-3 scores for transparency
    query = embed_frame(frames[len(frames)//2], model, preprocess, device)
    scores = {sid: (query @ p.T).item() for sid, p in PROTOTYPES.items()}
    top3   = sorted(scores.items(), key=lambda x: -x[1])[:3]
    print("\n  Top-3 candidates:")
    for rank, (sid, sc) in enumerate(top3, 1):
        label = VOCAB_LOOKUP[sid]
        bar   = "█" * int(sc * 30)
        print(f"    {rank}. {sid:<16} {label:<22} {sc:.3f}  {bar}")

    # ── TTS ────────────────────────────────────────────────────
    audio_path = speak(swahili_text, profile=USER_PROFILE, backend="gtts")
    display(Audio(audio_path, autoplay=True))
    print("─" * 52)


# Quick demo — classifies extracted frames from first sign
run_demo()


────────────────────────────────────────────────────
NENA · Tanzanian Sign Language · Inference Demo
────────────────────────────────────────────────────
ℹ️  Demo: using extracted frames from 'habari'
[Nena] Sign detected : habari
[Nena] Swahili text  : Habari
[Nena] Confidence    : 0.961

  Top-3 candidates:
    1. maji             Maji                   0.921  ███████████████████████████
    2. chakula          Chakula                0.912  ███████████████████████████
    3. jina_langu       Jina langu             0.909  ███████████████████████████
[Nena | gTTS  ] Text    : 'Habari'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmp9adri81p.mp3


────────────────────────────────────────────────────



# 9 — Webcam / video file inference loop

### In Colab, webcam access requires JavaScript bridge.
### This cell handles BOTH:
###   Option A: Video file upload (drag .mp4 into Colab sidebar)
###   Option B: Colab webcam snapshot (JavaScript bridge)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Video inference loop  (updated: clip-level, dedup, progress)
# ════════════════════════════════════════════════════════════════
from IPython.display import Audio, display
import cv2, numpy as np, time

# ── Option A: Process any of your 9 videos ────────────────────
def process_video_file(video_path, window_frames=10, step_frames=5, threshold=0.28):
    """
    Slides a window of `window_frames` over the video in steps of `step_frames`.
    Uses clip-level classifier on each window — much more stable than per-frame.
    Deduplicates consecutive identical predictions.

    Args:
        video_path    : path to .mp4 file
        window_frames : how many consecutive frames to classify together
        step_frames   : how many frames to advance between windows
        threshold     : cosine similarity cutoff (passed to classifier)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open: {video_path}")
        return

    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video : {video_path}")
    print(f"FPS   : {fps:.1f}  |  Total frames: {total}")
    print(f"Window: {window_frames} frames  |  Step: {step_frames} frames")
    print("─" * 52)

    all_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        all_frames.append(frame)
    cap.release()

    last_sign   = None
    predictions = []

    for start in range(0, len(all_frames) - window_frames + 1, step_frames):
        window  = all_frames[start : start + window_frames]
        sign_id, conf = classify_sign_from_clip(
            window, PROTOTYPES, model, preprocess, device, threshold=threshold
        )
        if sign_id and sign_id != last_sign:
            swahili = VOCAB_LOOKUP[sign_id]
            t_sec   = start / fps
            print(f"  t={t_sec:5.2f}s  frame {start:4d}  →  {sign_id:<16} {swahili:<22} [{conf:.3f}]")
            audio_path = speak(swahili, profile=USER_PROFILE, backend="gtts")
            display(Audio(audio_path, autoplay=False))
            predictions.append({"time": t_sec, "sign_id": sign_id,
                                 "swahili": swahili, "confidence": conf})
            last_sign = sign_id

    print(f"\n✅ Done. {len(predictions)} sign(s) detected.")
    return predictions


# ── Test on each of your 9 source videos ──────────────────────
print("Testing pipeline on all 9 source videos…\n")
for sign_id, vpath in VIDEO_MAP.items():
    import os
    if os.path.exists(vpath):
        print(f"\n▶  {sign_id.upper()}")
        process_video_file(vpath)
        print()


# ── Option B: Colab webcam snapshot ───────────────────────────
COLAB_WEBCAM_JS = """
async function capturePhoto() {
  const div   = document.createElement('div');
  const video = document.createElement('video');
  video.style.display = 'block';
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  document.body.appendChild(div);
  div.appendChild(video);
  video.srcObject = stream;
  await video.play();
  await new Promise(r => setTimeout(r, 2000));
  const canvas = document.createElement('canvas');
  canvas.width  = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);
  stream.getTracks().forEach(t => t.stop());
  div.remove();
  const dataUrl = canvas.toDataURL('image/jpeg', 0.8);
  google.colab.kernel.invokeFunction('notebook.photo_callback', [dataUrl], {});
}
capturePhoto();
"""

def capture_and_classify():
    try:
        from google.colab import output
        from IPython.display import Javascript, display
        import base64
        photo_data = {}
        def photo_callback(data_url):
            header, encoded = data_url.split(",", 1)
            photo_data["bytes"] = base64.b64decode(encoded)
        output.register_callback("notebook.photo_callback", photo_callback)
        display(Javascript(COLAB_WEBCAM_JS))
        time.sleep(4)
        if photo_data:
            nparr = np.frombuffer(photo_data["bytes"], np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            run_demo(frame_bgr=frame)
        else:
            print("⚠️  No photo captured.")
    except ImportError:
        print("⚠️  Not in Colab. Use process_video_file() instead.")

# capture_and_classify()   # ← Uncomment to trigger webcam


Testing pipeline on all 9 source videos…


▶  HABARI
Video : /content/drive/MyDrive/ZanAI/nena project/dataset/S0001_Habari.mp4
FPS   : 30.0  |  Total frames: 56
Window: 10 frames  |  Step: 5 frames
────────────────────────────────────────────────────
  t= 0.00s  frame    0  →  maji             Maji                   [0.937]
[Nena | gTTS  ] Text    : 'Maji'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpk7emeomo.mp3


  t= 0.17s  frame    5  →  habari           Habari                 [0.946]
[Nena | gTTS  ] Text    : 'Habari'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmphxd7sz03.mp3



✅ Done. 2 sign(s) detected.


▶  NZURI
Video : /content/drive/MyDrive/ZanAI/nena project/dataset/S0002_Nzuri.mp4
FPS   : 30.0  |  Total frames: 47
Window: 10 frames  |  Step: 5 frames
────────────────────────────────────────────────────
  t= 0.00s  frame    0  →  nzuri            Nzuri                  [0.941]
[Nena | gTTS  ] Text    : 'Nzuri'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmp6jr6rmtu.mp3



✅ Done. 1 sign(s) detected.


▶  ASANTE
Video : /content/drive/MyDrive/ZanAI/nena project/dataset/S0003_Asante.mp4
FPS   : 30.0  |  Total frames: 60
Window: 10 frames  |  Step: 5 frames
────────────────────────────────────────────────────
  t= 0.00s  frame    0  →  jina_langu       Jina langu             [0.939]
[Nena | gTTS  ] Text    : 'Jina langu'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmp5rhi6rp6.mp3


  t= 0.33s  frame   10  →  asante           Asante                 [0.947]
[Nena | gTTS  ] Text    : 'Asante'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpq1uqch8j.mp3


...

  t= 0.17s  frame   10  →  polisi           Polisi                 [0.897]
[Nena | gTTS  ] Text    : 'Polisi'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmp0f5j1zme.mp3


  t= 1.58s  frame   95  →  mfanyakazi       Mfanyakazi             [0.908]
[Nena | gTTS  ] Text    : 'Mfanyakazi'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpiwtxadkp.mp3



✅ Done. 3 sign(s) detected.


▶  MGENI
Video : /content/drive/MyDrive/ZanAI/nena project/dataset/S0070_Mgeni.mp4
FPS   : 60.0  |  Total frames: 208
Window: 10 frames  |  Step: 5 frames
────────────────────────────────────────────────────
  t= 0.00s  frame    0  →  polisi           Polisi                 [0.935]
[Nena | gTTS  ] Text    : 'Polisi'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmpzf7izw40.mp3


  t= 1.00s  frame   60  →  mgeni            Mgeni                  [0.918]
[Nena | gTTS  ] Text    : 'Mgeni'
[Nena | gTTS  ] Profile : Female (sw-TZ-Standard-A) — gender ignored by gTTS
[Nena | gTTS  ] Audio   → /tmp/tmphq9c02bx.mp3



✅ Done. 2 sign(s) detected.



# 10 — Save checkpoint & GitHub instructions


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — Save all artefacts for Nena app connection
#
# What to save and why:
#   nena_prototypes.pt    → the "model" — 9 prototype vectors (512-d each)
#   nena_vocabulary.json  → sign metadata (id, swahili, description, category)
#   nena_config.json      → inference config (threshold, model name, version)
#   nena_bundle.zip       → everything in one file to move to your app
#
# Your app (Dev C token relay API) calls the predictor with a frame/video
# and gets back: {"sign_id": "habari", "swahili": "Habari", "confidence": 0.82}
# ════════════════════════════════════════════════════════════════
import torch, json, zipfile, shutil, os
from pathlib import Path
from datetime import datetime

SAVE_DIR = Path("/content/nena_artefacts")
SAVE_DIR.mkdir(exist_ok=True)

# ── 1. Prototype embeddings (.pt) ─────────────────────────────
proto_path = SAVE_DIR / "nena_prototypes.pt"
torch.save(
    {sign_id: proto.cpu() for sign_id, proto in PROTOTYPES.items()},
    proto_path
)
print(f"✅ Prototypes    → {proto_path}")

# ── 2. Vocabulary JSON ────────────────────────────────────────
vocab_export = []
for sign in VOCABULARY:
    vocab_export.append({
        "id":          sign["id"],
        "swahili":     sign["swahili_text"],
        "description": sign["description"],
        "frame_count": len(sign.get("reference_img") or []),
    })
vocab_path = SAVE_DIR / "nena_vocabulary.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab_export, f, ensure_ascii=False, indent=2)
print(f"✅ Vocabulary    → {vocab_path}")

# ── 3. Inference config JSON ──────────────────────────────────
config = {
    "version":        "1.0.0",
    "built_at":       datetime.now().isoformat(),
    "clip_model":     "ViT-B-32",
    "clip_pretrained":"laion2b_s34b_b79k",
    "embed_dim":      512,
    "num_signs":      len(PROTOTYPES),
    "threshold":      0.28,
    "image_weight":   3.0,
    "signs":          [s["id"] for s in VOCABULARY],
    "api_response_schema": {
        "sign_id":    "str  — e.g. habari",
        "swahili":    "str  — e.g. Habari",
        "confidence": "float — cosine similarity 0-1",
        "top_3":      "list of above dicts"
    }
}
config_path = SAVE_DIR / "nena_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"✅ Config        → {config_path}")

# ── 4. Predictor script (standalone, no notebook needed) ──────
predictor_code = '''"""
nena_predictor.py — drop this file + nena_prototypes.pt + nena_vocabulary.json
into your backend and call predict_frame() or predict_video().

Requirements: pip install open-clip-torch opencv-python torch Pillow
"""
import json, torch, cv2, numpy as np
from PIL import Image
from pathlib import Path
import open_clip

# ── Load once at startup ──────────────────────────────────────
_MODEL, _PREPROCESS, _TOKENIZER, _PROTOTYPES, _VOCAB = None, None, None, None, None

def _load(artefact_dir="."):
    global _MODEL, _PREPROCESS, _TOKENIZER, _PROTOTYPES, _VOCAB
    if _MODEL is not None:
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _MODEL, _, _PREPROCESS = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k"
    )
    _MODEL = _MODEL.to(device).eval()
    _TOKENIZER = open_clip.get_tokenizer("ViT-B-32")

    raw = torch.load(f"{artefact_dir}/nena_prototypes.pt", map_location=device)
    _PROTOTYPES = {k: v.to(device) for k, v in raw.items()}

    with open(f"{artefact_dir}/nena_vocabulary.json") as f:
        _VOCAB = {s["id"]: s for s in json.load(f)}

def _embed(frame_bgr, device):
    rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    t    = _PREPROCESS(Image.fromarray(rgb)).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = _MODEL.encode_image(t)
        return feat / feat.norm(dim=-1, keepdim=True)

def predict_frame(frame_bgr, artefact_dir=".", threshold=0.28):
    """Single frame → sign prediction dict."""
    _load(artefact_dir)
    device = next(_MODEL.parameters()).device
    query  = _embed(frame_bgr, device)
    scores = {sid: (query @ p.T).item() for sid, p in _PROTOTYPES.items()}
    top3   = sorted(scores.items(), key=lambda x: -x[1])[:3]
    best_id, best_score = top3[0]
    if best_score < threshold:
        return {"sign_id": None, "swahili": None, "confidence": best_score, "top_3": []}
    result = lambda sid, sc: {"sign_id": sid, "swahili": _VOCAB[sid]["swahili"], "confidence": round(sc, 4)}
    return {**result(best_id, best_score), "top_3": [result(s, c) for s, c in top3]}

def predict_video(video_path, artefact_dir=".", threshold=0.28, n_frames=8):
    """Short video clip → sign prediction dict."""
    _load(artefact_dir)
    device = next(_MODEL.parameters()).device
    cap    = cv2.VideoCapture(video_path)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    idxs   = np.linspace(0, total-1, n_frames, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, f = cap.read()
        if ret: frames.append(f)
    cap.release()
    if not frames: return {"sign_id": None, "swahili": None, "confidence": 0.0, "top_3": []}
    feats  = [_embed(f, device) for f in frames]
    query  = torch.cat(feats, dim=0).mean(0, keepdim=True)
    query  = query / query.norm(dim=-1, keepdim=True)
    scores = {sid: (query @ p.T).item() for sid, p in _PROTOTYPES.items()}
    top3   = sorted(scores.items(), key=lambda x: -x[1])[:3]
    best_id, best_score = top3[0]
    if best_score < threshold:
        return {"sign_id": None, "swahili": None, "confidence": best_score, "top_3": []}
    result = lambda sid, sc: {"sign_id": sid, "swahili": _VOCAB[sid]["swahili"], "confidence": round(sc, 4)}
    return {**result(best_id, best_score), "top_3": [result(s, c) for s, c in top3]}
'''
pred_path = SAVE_DIR / "nena_predictor.py"
with open(pred_path, "w") as f:
    f.write(predictor_code)
print(f"✅ Predictor     → {pred_path}")

# ── 5. Bundle everything into one zip ─────────────────────────
bundle_path = "/content/nena_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in SAVE_DIR.iterdir():
        zf.write(fpath, fpath.name)
bundle_size = Path(bundle_path).stat().st_size / (1024*1024)
print(f"\n✅ Bundle        → {bundle_path}  ({bundle_size:.1f} MB)")

# ── 6. Copy bundle to Drive for persistence ───────────────────
drive_dest = "/content/drive/MyDrive/ZanAI/nena_bundle.zip"
try:
    import shutil
    os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
    shutil.copy(bundle_path, drive_dest)
    print(f"✅ Saved to Drive → {drive_dest}")
except Exception as e:
    print(f"⚠️  Drive save skipped: {e}")
    print(f"   Download manually: Files panel → /content/nena_bundle.zip → ⋮ → Download")

# ── 7. Summary ────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════════════╗
║           NENA — Artefacts Ready for App Connection          ║
╠══════════════════════════════════════════════════════════════╣
║  Files in nena_bundle.zip:                                   ║
║                                                              ║
║  nena_prototypes.pt    ← THE MODEL (512-d vectors per sign)  ║
║  nena_vocabulary.json  ← sign metadata (id, swahili, desc)   ║
║  nena_config.json      ← threshold, model name, version      ║
║  nena_predictor.py     ← drop-in predictor for backend       ║
║                                                              ║
║  How Dev C connects the app:                                 ║
║                                                              ║
║  1. Unzip bundle into backend folder                         ║
║  2. pip install open-clip-torch opencv-python torch Pillow   ║
║  3. In the token relay API:                                  ║
║       from nena_predictor import predict_frame, predict_video ║
║       result = predict_video("clip.mp4")                     ║
║       # result = {{"sign_id":"habari","swahili":"Habari",     ║
║       #            "confidence":0.82,"top_3":[...]}}          ║
║  4. Return result as JSON to the mobile app                  ║
╚══════════════════════════════════════════════════════════════╝
""")

# Download prompt
from google.colab import files
print("Downloading nena_bundle.zip to your computer…")
files.download(bundle_path)


✅ Prototypes    → /content/nena_artefacts/nena_prototypes.pt
✅ Vocabulary    → /content/nena_artefacts/nena_vocabulary.json
✅ Config        → /content/nena_artefacts/nena_config.json
✅ Predictor     → /content/nena_artefacts/nena_predictor.py

✅ Bundle        → /content/nena_bundle.zip  (0.1 MB)
✅ Saved to Drive → /content/drive/MyDrive/ZanAI/nena_bundle.zip

╔══════════════════════════════════════════════════════════════╗
║           NENA — Artefacts Ready for App Connection          ║
╠══════════════════════════════════════════════════════════════╣
║  Files in nena_bundle.zip:                                   ║
║                                                              ║
║  nena_prototypes.pt    ← THE MODEL (512-d vectors per sign)  ║
║  nena_vocabulary.json  ← sign metadata (id, swahili, desc)   ║
║  nena_config.json      ← threshold, model name, version      ║
║  nena_predictor.py     ← drop-in predictor for backend       ║
║                                                    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 11 — Evaluation: Per-sign accuracy report

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11 — Per-sign accuracy report
# Tests every sign's own video against the classifier and shows
# a clear pass/fail table + cosine score matrix.
# ════════════════════════════════════════════════════════════════
import numpy as np

def full_evaluation(video_map, prototypes, model, preprocess, device, threshold=0.28):
    """
    For each sign video:
      - samples 10 evenly-spaced frames
      - runs clip-level classify_sign_from_clip()
      - records predicted label + confidence

    Prints a full table and a mini cosine-similarity matrix.
    """
    print("\n── Per-sign accuracy ───────────────────────────────────────")
    print(f"  {'Sign ID':<16} {'True label':<22} {'Predicted':<22} {'Conf':>6}  {'Result'}")
    print(f"  {'─'*16} {'─'*22} {'─'*22} {'─'*6}  {'─'*6}")

    correct, total = 0, 0
    all_scores = {}   # sign_id → {sign_id: score} for matrix

    for true_id, vpath in video_map.items():
        import os
        if not os.path.exists(vpath):
            continue

        cap     = cv2.VideoCapture(vpath)
        n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        indices = np.linspace(0, n_total-1, 10, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, f = cap.read()
            if ret: frames.append(f)
        cap.release()

        pred_id, conf = classify_sign_from_clip(
            frames, prototypes, model, preprocess, device, threshold=threshold
        )

        # Row scores for matrix
        if frames:
            feats = [embed_frame(f, model, preprocess, device) for f in frames]
            query = torch.cat(feats).mean(0, keepdim=True)
            query = query / query.norm(dim=-1, keepdim=True)
            all_scores[true_id] = {sid: (query @ p.T).item() for sid, p in prototypes.items()}

        true_label = VOCAB_LOOKUP[true_id]
        pred_label = VOCAB_LOOKUP.get(pred_id, "—") if pred_id else "below threshold"
        ok     = pred_id == true_id
        symbol = "✅" if ok else "❌"
        print(f"  {true_id:<16} {true_label:<22} {pred_label:<22} {conf:>6.3f}  {symbol}")
        correct += int(ok)
        total   += 1

    acc = correct / total * 100 if total else 0
    print(f"\n  ── Accuracy: {correct}/{total} = {acc:.0f}% ──")

    if acc < 70:
        print("  ℹ️  Below 70% — expected with 1 video/sign.")
        print("      Record 3–5 more videos per sign to improve significantly.")
    elif acc < 90:
        print("  ℹ️  Good for a 1-video prototype. Add more signers to reach 90%+.")
    else:
        print("  🎉 Excellent prototype accuracy!")

    # Mini similarity matrix
    if all_scores:
        signs = list(all_scores.keys())
        print("\n── Cosine similarity matrix (diagonal = self-score, want high) ──")
        header = "".join(f"{s[:7]:>8}" for s in signs)
        print(f"  {'':>14} {header}")
        for true_id in signs:
            row = "".join(
                f"\033[92m{all_scores[true_id].get(s,0):>8.3f}\033[0m"
                if s == true_id
                else f"{all_scores[true_id].get(s,0):>8.3f}"
                for s in signs
            )
            print(f"  {true_id:<14} {row}")

full_evaluation(VIDEO_MAP, PROTOTYPES, model, preprocess, device)



── Per-sign accuracy ───────────────────────────────────────
  Sign ID          True label             Predicted                Conf  Result
  ──────────────── ────────────────────── ────────────────────── ──────  ──────
  habari           Habari                 Habari                  0.956  ✅
  nzuri            Nzuri                  Nzuri                   0.958  ✅
  asante           Asante                 Asante                  0.958  ✅
  mbaya            Mbaya                  Mbaya                   0.958  ✅
  maji             Maji                   Maji                    0.957  ✅
  tafadhali_au_samahani Tafadhali au Samahani  Tafadhali au Samahani   0.961  ✅
  chakula          Chakula                Chakula                 0.953  ✅
  jina_langu       Jina langu             Jina langu              0.958  ✅
  kwaheri          Kwaheri                Kwaheri                 0.959  ✅
  kula             Kula                   Kula                    0.959  ✅
  kunywa           Kuny

# 12 — How to add more signs (expanding beyond 9)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 12 — Add a new sign to Nena (template)
#
# Use this cell every time you record a new LAT sign.
# Steps:
#   1. Record the sign video on your phone
#   2. Upload it to Colab via files.upload()
#   3. Fill in the new_sign dict below
#   4. Run this cell → prototypes auto-update
#   5. Re-run Cell 10 to save the updated bundle
# ════════════════════════════════════════════════════════════════
from google.colab import files as colab_files

def add_new_sign(new_sign: dict, video_path: str, fps_extract=3.0, max_frames=15):
    """
    new_sign dict keys: id, swahili_text, description
    video_path: path to the .mp4 after upload
    """
    # Extract frames
    sign_dir = FRAME_DIR / new_sign["id"]
    sign_dir.mkdir(parents=True, exist_ok=True)
    cap   = cv2.VideoCapture(video_path)
    vfps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step  = max(1, int(vfps / fps_extract))
    saved = []
    fi    = 0
    while len(saved) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if fi % step == 0:
            p = sign_dir / f"frame_{len(saved):04d}.jpg"
            cv2.imwrite(str(p), frame)
            saved.append(str(p))
        fi += 1
    cap.release()
    new_sign["reference_img"] = saved
    print(f"  ✅ {new_sign['id']} — {len(saved)} frames extracted")

    # Add to vocabulary
    VOCABULARY.append(new_sign)
    VIDEO_MAP[new_sign["id"]] = video_path

    # Rebuild only the new prototype (fast — doesn't recompute all)
    import torch
    with torch.no_grad():
        prompts = [
            new_sign["description"],
            f"A sign language gesture: {new_sign['description']}",
            f"Tanzanian sign language sign meaning {new_sign['swahili_text']}: {new_sign['description']}",
        ]
        tokens     = tokenizer(prompts).to(device)
        text_feats = model.encode_text(tokens)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        text_proto = text_feats.mean(0, keepdim=True)
        text_proto = text_proto / text_proto.norm(dim=-1, keepdim=True)

        if len(saved) >= 3:
            img_feats = []
            for p in saved:
                img  = preprocess(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
                feat = model.encode_image(img)
                img_feats.append(feat / feat.norm(dim=-1, keepdim=True))
            img_proto = torch.cat(img_feats).mean(0, keepdim=True)
            img_proto = img_proto / img_proto.norm(dim=-1, keepdim=True)
            copies    = [img_proto]*3 + [text_proto]
            proto     = torch.cat(copies).mean(0, keepdim=True)
            proto     = proto / proto.norm(dim=-1, keepdim=True)
        else:
            proto = text_proto

    PROTOTYPES[new_sign["id"]] = proto
    VOCAB_LOOKUP[new_sign["id"]] = new_sign["swahili_text"]
    print(f"  ✅ Prototype built — PROTOTYPES now has {len(PROTOTYPES)} signs")
    print(f"  ⚡ Re-run Cell 10 to save the updated bundle!")


# ── EXAMPLE: adding "Ndiyo" (Yes) ─────────────────────────────
# 1. Upload your video first:
# uploaded = colab_files.upload()   # upload S0010_Ndiyo.mp4

# 2. Then call add_new_sign():
# add_new_sign(
#     new_sign={
#         "id": "ndiyo",
#         "swahili_text": "Ndiyo",
#         "description": "A person nodding the head or raising a fist and moving it downward to indicate yes in Tanzanian sign language",
#     },
#     video_path="/content/S0010_Ndiyo.mp4"
# )

print("Cell 12 ready. Uncomment the add_new_sign() call above to add a new sign.")
print(f"Current vocabulary: {len(VOCABULARY)} signs — {[s['id'] for s in VOCABULARY]}")


Cell 12 ready. Uncomment the add_new_sign() call above to add a new sign.
Current vocabulary: 60 signs — ['habari', 'nzuri', 'asante', 'mbaya', 'maji', 'tafadhali_au_samahani', 'chakula', 'jina_langu', 'kwaheri', 'kula', 'kunywa', 'kulala', 'kusoma', 'kuandika', 'kutembea', 'kukimbia', 'kufanya_kazi', 'usafiri', 'kucheza', 'kuimba', 'kuona', 'kusikia', 'kununua', 'kuuza', 'kupenda', 'kusaidia', 'kujifunza', 'dada', 'kaka', 'mjomba', 'shangazi', 'bibi', 'mama', 'baba', 'familia', 'babu', 'mke', 'mume', 'mtoto', 'vipi', 'kwanini', 'lini', 'wapi', 'nini', 'wao', 'nyinyi', 'sisi', 'yeye', 'wewe', 'mimi', 'rafiki', 'mwanafunzi', 'mwalimu', 'daktari', 'muuguzi', 'dereva', 'polisi', 'mkuu', 'mfanyakazi', 'mgeni']


# 13 NENA API Server — LIVE

In [ ]:
!pip install -q flask pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("3EBmtqhRWMhngG50pYvCLCGDRLE_7PNCUMwaUgXRGcBRJfA4r")

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║  NENA — Colab API Server                                                 ║
║                                                                          ║
║  Paste this as the LAST cell in your Colab notebook after Cell 10.      ║
║                                                                          ║
║  What this does:                                                         ║
║    1. Loads nena_prototypes.pt + nena_vocabulary.json from your bundle   ║
║    2. Starts a Flask server with /predict and /health endpoints          ║
║    3. Exposes it publicly via ngrok (free, no account needed)            ║
║    4. Prints the URL → paste it into ai_backend_service.dart             ║
╚══════════════════════════════════════════════════════════════════════════╝
"""

# ════════════════════════════════════════════════════════════════════════
# CELL 13 — Install server dependencies (run once per session)
# ════════════════════════════════════════════════════════════════════════
"""
!pip install -q flask pyngrok
"""

# ════════════════════════════════════════════════════════════════════════
# CELL 14 — Flask API server
#
# Endpoints:
#   GET  /health          → {"status": "ok", "signs": 9}
#   POST /predict         → {"frame": "<base64 jpeg>"} → prediction JSON
#   POST /predict_clip    → {"frames": ["<b64>", ...]} → prediction JSON  (optional)
#
# Response format (matches what ai_backend_service.dart already expects):
#   {
#     "prediction":  "Habari",          ← swahili_text (for TTS)
#     "sign_id":     "habari",          ← internal ID
#     "confidence":  0.87,
#     "top_3": [
#       {"sign_id": "habari",  "swahili": "Habari",  "confidence": 0.87},
#       {"sign_id": "kwaheri", "swahili": "Kwaheri", "confidence": 0.21},
#       {"sign_id": "nzuri",   "swahili": "Nzuri",   "confidence": 0.18}
#     ]
#   }
# ════════════════════════════════════════════════════════════════════════

import base64
import json
import threading
import numpy as np
import cv2
import torch
from PIL import Image
from io import BytesIO
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

# ── Model state (loaded once, reused for every request) ──────────────────
# These are already in memory from your earlier cells (PROTOTYPES, VOCAB_LOOKUP,
# model, preprocess, device). The server just wraps them in HTTP endpoints.
# If you restart the kernel, re-run cells 1-10 first, then run this cell.

THRESHOLD = 0.28    # Cosine similarity threshold from your notebook


# ════════════════════════════════════════════════════════════════════════
# Helper: decode a base64 JPEG string → OpenCV BGR frame
# ════════════════════════════════════════════════════════════════════════

def b64_to_frame(b64_string: str) -> np.ndarray:
    """Decodes a base64-encoded JPEG string into an OpenCV BGR numpy array."""
    img_bytes = base64.b64decode(b64_string)
    nparr     = np.frombuffer(img_bytes, dtype=np.uint8)
    frame     = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return frame


# ════════════════════════════════════════════════════════════════════════
# Helper: embed frame(s) and score against prototypes
# ════════════════════════════════════════════════════════════════════════

def _embed_frame(frame_bgr: np.ndarray) -> torch.Tensor:
    """BGR frame → normalised CLIP embedding (1, 512)."""
    rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    t    = preprocess(Image.fromarray(rgb)).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(t)
        return feat / feat.norm(dim=-1, keepdim=True)


def _score_query(query: torch.Tensor) -> dict:
    """
    Score a query embedding against all prototypes.
    Returns a prediction dict ready to send as JSON.
    """
    scores  = {sid: (query @ p.T).item() for sid, p in PROTOTYPES.items()}
    top3    = sorted(scores.items(), key=lambda x: -x[1])[:3]
    best_id, best_score = top3[0]

    def fmt(sid, sc):
        return {
            "sign_id":    sid,
            "swahili":    VOCAB_LOOKUP.get(sid, sid),
            "confidence": round(sc, 4),
        }

    if best_score < THRESHOLD:
        return {
            "prediction":  None,
            "sign_id":     None,
            "confidence":  round(best_score, 4),
            "top_3":       [fmt(s, c) for s, c in top3],
        }

    return {
        "prediction":  VOCAB_LOOKUP.get(best_id, best_id),  # Swahili text for TTS
        "sign_id":     best_id,
        "confidence":  round(best_score, 4),
        "top_3":       [fmt(s, c) for s, c in top3],
    }


# ════════════════════════════════════════════════════════════════════════
# Routes
# ════════════════════════════════════════════════════════════════════════

@app.route("/health", methods=["GET"])
def health():
    """
    Called by AiBackendService.isBackendReachable() in your Flutter app.
    Returns 200 OK when the model is loaded and ready.
    """
    return jsonify({
        "status": "ok",
        "signs":  len(PROTOTYPES),
        "model":  "CLIP ViT-B/32",
        "sign_ids": list(PROTOTYPES.keys()),
    })


@app.route("/predict", methods=["POST"])
def predict():
    """
    Main prediction endpoint.
    Called by AiBackendService.predictSign() every 500ms from the app.

    Request body (JSON):
        {"frame": "<base64-encoded JPEG string>"}

    Response (JSON):
        {
          "prediction":  "Habari",
          "sign_id":     "habari",
          "confidence":  0.87,
          "top_3":       [...]
        }
    """
    data = request.get_json(force=True)

    if "frame" not in data:
        return jsonify({"error": "Missing 'frame' field"}), 400

    try:
        frame = b64_to_frame(data["frame"])
        if frame is None:
            return jsonify({"error": "Could not decode image"}), 400

        query  = _embed_frame(frame)
        result = _score_query(query)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/predict_clip", methods=["POST"])
def predict_clip():
    """
    Optional: send multiple frames from a short video clip.
    Averages the embeddings for a more stable prediction.

    Request body (JSON):
        {"frames": ["<b64>", "<b64>", ...]}   ← 4-8 frames recommended
    """
    data = request.get_json(force=True)

    if "frames" not in data or not data["frames"]:
        return jsonify({"error": "Missing 'frames' list"}), 400

    try:
        feats = []
        for b64 in data["frames"]:
            frame = b64_to_frame(b64)
            if frame is not None:
                feats.append(_embed_frame(frame))

        if not feats:
            return jsonify({"error": "No valid frames decoded"}), 400

        # Average embeddings across frames → more stable than single frame
        query = torch.cat(feats, dim=0).mean(dim=0, keepdim=True)
        query = query / query.norm(dim=-1, keepdim=True)

        result = _score_query(query)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# ════════════════════════════════════════════════════════════════════════
# Start server + ngrok tunnel
# ════════════════════════════════════════════════════════════════════════

def start_server():
    """Run Flask in a background thread (so the Colab cell doesn't block)."""
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)


# Start Flask in background thread
t = threading.Thread(target=start_server, daemon=True)
t.start()

# Open ngrok tunnel
public_url = ngrok.connect(5000, bind_tls=True).public_url

print("╔══════════════════════════════════════════════════════════════╗")
print("║        NENA API Server — LIVE                                ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Public URL: {public_url:<48}║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║                                                              ║")
print("║  STEP: Copy the URL above and paste it into your app:       ║")
print("║                                                              ║")
print("║  File: lib/services/ai_backend_service.dart                  ║")
print("║  Line: static const String _baseUrl = 'PASTE_HERE';         ║")
print("║                                                              ║")
print("║  Test the server right now in your browser:                  ║")
print(f"║  {public_url}/health")
print("║                                                              ║")
print("║  ⚠  Keep this Colab tab open — closing it kills the server  ║")
print("╚══════════════════════════════════════════════════════════════╝")

# Quick self-test
import requests, time
time.sleep(1.5)
try:
    r = requests.get(f"{public_url}/health", timeout=5)
    d = r.json()
    print(f"\n✅ Self-test passed — {d['signs']} signs loaded: {d['sign_ids']}")
except Exception as e:
    print(f"\n⚠  Self-test: {e}  (server may still be starting — try the URL in your browser)")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


╔══════════════════════════════════════════════════════════════╗
║        NENA API Server — LIVE                                ║
╠══════════════════════════════════════════════════════════════╣
║  Public URL: https://diffuser-escalate-efficient.ngrok-free.dev║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  STEP: Copy the URL above and paste it into your app:       ║
║                                                              ║
║  File: lib/services/ai_backend_service.dart                  ║
║  Line: static const String _baseUrl = 'PASTE_HERE';         ║
║                                                              ║
║  Test the server right now in your browser:                  ║
║  https://diffuser-escalate-efficient.ngrok-free.dev/health
║                                                              ║
║  ⚠  Keep this Colab tab open — closing it kills the server  ║
╚═════════════════════════════

INFO:werkzeug:127.0.0.1 - - [30/Aug/2026 10:19:29] "GET /health HTTP/1.1" 200 -



✅ Self-test passed — 60 signs loaded: ['habari', 'nzuri', 'asante', 'mbaya', 'maji', 'tafadhali_au_samahani', 'chakula', 'jina_langu', 'kwaheri', 'kula', 'kunywa', 'kulala', 'kusoma', 'kuandika', 'kutembea', 'kukimbia', 'kufanya_kazi', 'usafiri', 'kucheza', 'kuimba', 'kuona', 'kusikia', 'kununua', 'kuuza', 'kupenda', 'kusaidia', 'kujifunza', 'dada', 'kaka', 'mjomba', 'shangazi', 'bibi', 'mama', 'baba', 'familia', 'babu', 'mke', 'mume', 'mtoto', 'vipi', 'kwanini', 'lini', 'wapi', 'nini', 'wao', 'nyinyi', 'sisi', 'yeye', 'wewe', 'mimi', 'rafiki', 'mwanafunzi', 'mwalimu', 'daktari', 'muuguzi', 'dereva', 'polisi', 'mkuu', 'mfanyakazi', 'mgeni']


In [ ]:
import base64
import json
import threading
import subprocess
import re
import time
import numpy as np
import cv2
import torch
from PIL import Image
from flask import Flask, request, jsonify

app = Flask(__name__)
THRESHOLD = 0.28

# ── Helpers ───────────────────────────────────────────────────────────────

def b64_to_frame(b64_string):
    img_bytes = base64.b64decode(b64_string)
    nparr     = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(nparr, cv2.IMREAD_COLOR)

def _embed_frame(frame_bgr):
    rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    t    = preprocess(Image.fromarray(rgb)).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(t)
        return feat / feat.norm(dim=-1, keepdim=True)

def _score_query(query):
    scores  = {sid: (query @ p.T).item() for sid, p in PROTOTYPES.items()}
    top3    = sorted(scores.items(), key=lambda x: -x[1])[:3]
    best_id, best_score = top3[0]
    def fmt(sid, sc):
        return {"sign_id": sid, "swahili": VOCAB_LOOKUP.get(sid, sid), "confidence": round(sc, 4)}
    if best_score < THRESHOLD:
        return {"prediction": None, "sign_id": None, "confidence": round(best_score, 4), "top_3": [fmt(s,c) for s,c in top3]}
    return {"prediction": VOCAB_LOOKUP.get(best_id, best_id), "sign_id": best_id, "confidence": round(best_score, 4), "top_3": [fmt(s,c) for s,c in top3]}

# ── Routes ────────────────────────────────────────────────────────────────

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "signs": len(PROTOTYPES), "model": "CLIP ViT-B/32", "sign_ids": list(PROTOTYPES.keys())})

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(force=True)
    if "frame" not in data:
        return jsonify({"error": "Missing 'frame' field"}), 400
    try:
        frame = b64_to_frame(data["frame"])
        if frame is None:
            return jsonify({"error": "Could not decode image"}), 400
        return jsonify(_score_query(_embed_frame(frame)))
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/predict_clip", methods=["POST"])
def predict_clip():
    data = request.get_json(force=True)
    if "frames" not in data or not data["frames"]:
        return jsonify({"error": "Missing 'frames' list"}), 400
    try:
        feats = [_embed_frame(b64_to_frame(b)) for b in data["frames"] if b64_to_frame(b) is not None]
        if not feats:
            return jsonify({"error": "No valid frames"}), 400
        query = torch.cat(feats, dim=0).mean(dim=0, keepdim=True)
        query = query / query.norm(dim=-1, keepdim=True)
        return jsonify(_score_query(query))
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ── Start Flask ───────────────────────────────────────────────────────────

def start_server():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)
print("✅ Flask running on port 5000")

# ── Cloudflare Tunnel (no account, no token needed) ───────────────────────

# Install cloudflared
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True, check=True
)

# Start tunnel in background, capture output to find the URL
tunnel_log = "/tmp/cloudflared.log"
subprocess.Popen(
    f"cloudflared tunnel --url http://localhost:5000 --no-autoupdate > {tunnel_log} 2>&1",
    shell=True
)

# Wait for the public URL to appear in the log (usually 5-10 seconds)
print("Starting Cloudflare tunnel", end="")
public_url = None
for _ in range(30):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        log = open(tunnel_log).read()
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", log)
        if match:
            public_url = match.group(0)
            break
    except:
        pass

print()  # newline after dots

if public_url:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║        NENA API Server — LIVE                                ║")
    print("╠══════════════════════════════════════════════════════════════╣")
    print(f"║  Public URL:  {public_url}")
    print("╠══════════════════════════════════════════════════════════════╣")
    print("║                                                              ║")
    print("║  Paste this URL into your Flutter app:                       ║")
    print("║  lib/services/ai_backend_service.dart                        ║")
    print("║  static const String _baseUrl = 'PASTE_URL_HERE';           ║")
    print("║                                                              ║")
    print(f"║  Test in browser: {public_url}/health")
    print("║                                                              ║")
    print("║  ⚠  Keep this Colab tab open — closing kills the tunnel     ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    # Self-test
    import requests as req
    try:
        r = req.get(f"{public_url}/health", timeout=10)
        d = r.json()
        print(f"\n✅ Self-test passed — {d['signs']} signs ready: {d['sign_ids']}")
    except Exception as e:
        print(f"\n⚠  Self-test: {e} — try opening the /health URL in your browser manually")
else:
    print("⚠  Could not get tunnel URL. Check the log:")
    print(open(tunnel_log).read()[-1000:])

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


✅ Flask running on port 5000
Starting Cloudflare tunnel....
╔══════════════════════════════════════════════════════════════╗
║        NENA API Server — LIVE                                ║
╠══════════════════════════════════════════════════════════════╣
║  Public URL:  https://builds-planners-position-garlic.trycloudflare.com
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Paste this URL into your Flutter app:                       ║
║  lib/services/ai_backend_service.dart                        ║
║  static const String _baseUrl = 'PASTE_URL_HERE';           ║
║                                                              ║
║  Test in browser: https://builds-planners-position-garlic.trycloudflare.com/health
║                                                              ║
║  ⚠  Keep this Colab tab open — closing kills the tunnel     ║
╚══════════════════════════════════════════════════════════════╝

⚠ 

# Speech-to-Avatar inbound pipeline

### Step 1 — Install Whisper

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║  NENA — Speech-to-Avatar Backend  (Stream B: Inbound Pipeline)          ║
║                                                                          ║
║  Paste this as new cells in your existing Colab notebook                ║
║  (after Cell 10, alongside your existing Flask server cells)            ║
║                                                                          ║
║  What this adds:                                                         ║
║    POST /transcribe  → accepts caller audio → returns Swahili text      ║
║                        which the Flutter AvatarWidget signs on screen   ║
║                                                                          ║
║  Full pipeline:                                                          ║
║    Caller speaks → mic captures PCM/WAV bytes                           ║
║    Flutter WhisperService sends base64 audio → POST /transcribe         ║
║    Whisper ASR decodes → Swahili text                                   ║
║    Keyword extractor strips fillers → sign tokens                       ║
║    Response → AvatarScreen → AvatarWidget animates each sign            ║
╚══════════════════════════════════════════════════════════════════════════╝

"""


'\n╔══════════════════════════════════════════════════════════════════════════╗\n║  NENA — Speech-to-Avatar Backend  (Stream B: Inbound Pipeline)          ║\n║                                                                          ║\n║  Paste this as new cells in your existing Colab notebook                ║\n║  (after Cell 10, alongside your existing Flask server cells)            ║\n║                                                                          ║\n║  What this adds:                                                         ║\n║    POST /transcribe  → accepts caller audio → returns Swahili text      ║\n║                        which the Flutter AvatarWidget signs on screen   ║\n║                                                                          ║\n║  Full pipeline:                                                          ║\n║    Caller speaks → mic captures PCM/WAV bytes                           ║\n║    Flutter WhisperService sends base64 audio → POST /transcribe  

# CELL 15 — Install dependencies (run once per Colab session)

In [ ]:
!pip install -q openai-whisper flask pyngrok
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# CELL 16 — Load Whisper model


In [ ]:
import whisper
import torch

# "small" is the sweet-spot for Swahili on Colab T4:
#   · ~244 MB VRAM  (fits easily alongside CLIP)
#   · ~3–5× realtime speed on T4 GPU
#   · Good accuracy on Swahili/East-African accents
#   · Change to "medium" if accuracy matters more than speed
WHISPER_MODEL_SIZE = "small"

print(f"Loading Whisper '{WHISPER_MODEL_SIZE}' … (downloads ~244 MB on first run)")
_whisper_device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model   = whisper.load_model(WHISPER_MODEL_SIZE, device=_whisper_device)
print(f"✅ Whisper '{WHISPER_MODEL_SIZE}' ready on {_whisper_device.upper()}")


Loading Whisper 'small' … (downloads ~244 MB on first run)


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 93.4MiB/s]


✅ Whisper 'small' ready on CUDA


# CELL 17 — Swahili keyword extractor

In [ ]:
# Called AFTER transcription. Strips grammatical glue words so the avatar
# only signs semantically meaningful tokens, matching the logic already
# in your WhisperService.dart.
# ════════════════════════════════════════════════════════════════════════════
import re

# Swahili filler / connector words — matches the set in whisper_service.dart
_SWAHILI_FILLERS = {
    "na", "ya", "wa", "la", "kwa", "ni", "si", "au",
    "pia", "tu", "hii", "hiyo", "ile", "hizo",
    "a", "e", "i", "o", "u",
    # Additional common particles
    "katika", "kama", "lakini", "ama", "bali", "nk",
    "hivyo", "hata", "sana", "zaidi", "kidogo",
}

def extract_keywords(sentence: str) -> list[str]:
    """
    Return a list of meaningful sign tokens from a Swahili sentence.
    Strips punctuation, lowercases, and removes filler words.

    Examples
    --------
    >>> extract_keywords("Habari yako, una hali gani?")
    ['habari', 'yako', 'una', 'hali', 'gani']

    >>> extract_keywords("Asante sana kwa msaada wako")
    ['asante', 'msaada', 'wako']
    """
    cleaned = re.sub(r"[^\w\s]", "", sentence.lower())
    tokens  = cleaned.split()
    return [t for t in tokens if len(t) > 1 and t not in _SWAHILI_FILLERS]


# CELL 18 — Audio decoder

In [ ]:
# Flutter sends audio bytes as base64.  The bytes can be:
#   · Raw 16-bit PCM at 16 kHz (from flutter_sound recorder)
#   · A WAV container (most common)
#   · An M4A/AAC chunk (less common; ffmpeg handles it automatically)
#
# We write to a temp file so Whisper's ffmpeg loader can handle any format.
# ════════════════════════════════════════════════════════════════════════════
import base64
import tempfile
import os

def decode_audio_to_tempfile(b64_audio: str) -> str:
    """
    Decode a base64 audio string → write to a named temp file.
    Returns the temp file path.  Caller must delete it when done.
    Supports WAV, PCM, M4A/AAC, OGG, MP3 — ffmpeg handles them all.
    """
    raw_bytes = base64.b64decode(b64_audio)

    # Detect format from magic bytes
    if raw_bytes[:4] == b"RIFF":
        suffix = ".wav"
    elif raw_bytes[:3] == b"ID3" or raw_bytes[:2] == b"\xff\xfb":
        suffix = ".mp3"
    elif raw_bytes[4:8] == b"ftyp":
        suffix = ".m4a"
    else:
        # Assume raw PCM — wrap in a proper WAV so ffmpeg can decode it
        suffix = ".wav"
        raw_bytes = _pcm_to_wav(raw_bytes, sample_rate=16000, channels=1, bit_depth=16)

    tmp = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
    tmp.write(raw_bytes)
    tmp.close()
    return tmp.name


def _pcm_to_wav(pcm_bytes: bytes, sample_rate=16000, channels=1, bit_depth=16) -> bytes:
    """
    Wrap raw signed-integer PCM in a minimal WAV header.
    Flutter's flutter_sound records at 16 kHz / 16-bit / mono by default.
    """
    import struct
    data_size   = len(pcm_bytes)
    byte_rate   = sample_rate * channels * (bit_depth // 8)
    block_align = channels * (bit_depth // 8)

    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF",
        36 + data_size,   # ChunkSize
        b"WAVE",
        b"fmt ",
        16,               # Subchunk1Size (PCM)
        1,                # AudioFormat (1 = PCM)
        channels,
        sample_rate,
        byte_rate,
        block_align,
        bit_depth,
        b"data",
        data_size,
    )
    return header + pcm_bytes

# CELL 19 — /transcribe Flask endpoint

In [ ]:
# Merge this app into your existing Flask `app` object from Cell 14.
# If you run this as a fresh server, define `app = Flask(__name__)` here.
#
# Request  (JSON):
#   { "audio": "<base64-encoded audio bytes>" }
#
# Response (JSON):
#   {
#     "text":     "Habari yako, una hali gani?",   ← full Whisper transcript
#     "keywords": ["habari", "yako", "una", "hali", "gani"],  ← sign tokens
#     "language": "sw",                            ← detected language code
#     "duration": 2.4                              ← audio clip length in secs
#   }
#
# Flutter side: WhisperService.transcribeAudio() already calls this endpoint.
# AvatarScreen iterates over `keywords` and calls _onWordReceived() per token.
# ════════════════════════════════════════════════════════════════════════════

# NOTE: This `app` assumes you already defined it in Cell 14.
# If running standalone: uncomment the next line.
# app = Flask(__name__)

from flask import Flask, request, jsonify   # noqa: F811  (already imported in Cell 14)

@app.route("/transcribe", methods=["POST"])
def transcribe():
    """
    Transcribe a caller's voice clip to Swahili text and sign tokens.

    Called every ~2 seconds by WhisperService.transcribeAudio() as the
    caller speaks. The returned `keywords` list drives AvatarWidget sign
    animations on the Deaf user's screen.
    """
    data = request.get_json(force=True)

    if "audio" not in data:
        return jsonify({"error": "Missing 'audio' field (base64-encoded bytes)"}), 400

    tmp_path = None
    try:
        # ── 1. Decode base64 audio to temp file ──────────────────────────
        tmp_path = decode_audio_to_tempfile(data["audio"])

        # ── 2. Transcribe with Whisper ────────────────────────────────────
        # `language="sw"` locks to Swahili and prevents language detection
        # errors on short clips. Remove to allow auto-detect.
        result = whisper_model.transcribe(
            tmp_path,
            language="sw",             # Swahili
            task="transcribe",         # Keep original language (not translate)
            fp16=(_whisper_device == "cuda"),
            condition_on_previous_text=False,  # Treat each clip independently
            temperature=0.0,           # Greedy decode — faster and deterministic
            no_speech_threshold=0.5,   # Skip near-silent clips
            logprob_threshold=-1.0,    # Accept low-confidence tokens (short clips)
        )

        full_text = result.get("text", "").strip()
        language  = result.get("language", "sw")

        # Duration from audio segments (fallback: estimate from file size)
        segs     = result.get("segments", [])
        duration = segs[-1]["end"] if segs else 0.0

        # ── 3. Extract sign tokens ────────────────────────────────────────
        keywords = extract_keywords(full_text)

        return jsonify({
            "text":     full_text,
            "keywords": keywords,
            "language": language,
            "duration": round(duration, 2),
        })

    except Exception as exc:
        return jsonify({"error": str(exc)}), 500

    finally:
        # Always clean up the temp audio file
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)

# CELL 20 — Optional: /transcribe_stream  (word-by-word streaming variant)

In [ ]:
# A more responsive variant: transcribes the clip, then streams each keyword
# back one-by-one with a small delay so the avatar can sign words as they
# arrive rather than waiting for the full clip to process.
#
# Flutter side: use http SSE (Server-Sent Events) or poll /transcribe_stream
# and call _onWordReceived() for each word chunk.
# This endpoint is OPTIONAL — /transcribe above is simpler and works well.
# ════════════════════════════════════════════════════════════════════════════
import time
from flask import Response, stream_with_context
import json as _json

@app.route("/transcribe_stream", methods=["POST"])
def transcribe_stream():
    """
    Same as /transcribe but streams each keyword as a Server-Sent Event (SSE).

    Flutter usage (advanced):
        final request = http.Request('POST', uri);
        request.body = jsonEncode({'audio': b64Audio});
        final response = await client.send(request);
        await for (final chunk in response.stream) {
          final word = utf8.decode(chunk).split('data: ')[1].trim();
          _onWordReceived(word);
        }
    """
    data = request.get_json(force=True)

    if "audio" not in data:
        return jsonify({"error": "Missing 'audio' field"}), 400

    tmp_path = None

    def generate():
        nonlocal tmp_path
        try:
            tmp_path = decode_audio_to_tempfile(data["audio"])

            result   = whisper_model.transcribe(
                tmp_path,
                language="sw",
                task="transcribe",
                fp16=(_whisper_device == "cuda"),
                condition_on_previous_text=False,
                temperature=0.0,
                no_speech_threshold=0.5,
            )

            full_text = result.get("text", "").strip()
            keywords  = extract_keywords(full_text)

            # Stream each keyword as a separate SSE event
            for kw in keywords:
                payload = _json.dumps({"word": kw, "full_text": full_text})
                yield f"data: {payload}\n\n"
                time.sleep(0.15)   # Small gap between words

            # Final marker so Flutter knows the stream ended
            yield f"data: {_json.dumps({'done': True, 'total_keywords': len(keywords)})}\n\n"

        except Exception as exc:
            yield f"data: {_json.dumps({'error': str(exc)})}\n\n"
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.unlink(tmp_path)

    return Response(
        stream_with_context(generate()),
        mimetype="text/event-stream",
        headers={
            "Cache-Control":               "no-cache",
            "X-Accel-Buffering":           "no",
            "Access-Control-Allow-Origin": "*",
        },
    )

# CELL 21 — /health update

In [ ]:
# Update your existing /health route to confirm Whisper is loaded.
# Replace or extend the existing health() function:
# ════════════════════════════════════════════════════════════════════════════

# Uncomment and replace the health() function in Cell 14 with this:
#
# @app.route("/health", methods=["GET"])
# def health():
#     return jsonify({
#         "status":         "ok",
#         "signs":          len(PROTOTYPES),          # from Cell 10
#         "clip_model":     "CLIP ViT-B/32",
#         "whisper_model":  WHISPER_MODEL_SIZE,
#         "sign_ids":       list(PROTOTYPES.keys()),
#         "endpoints": {
#             "/predict":            "POST  sign frame → Swahili text (Sign→Speech)",
#             "/transcribe":         "POST  audio clip → sign tokens  (Speech→Avatar)",
#             "/transcribe_stream":  "POST  audio clip → SSE keyword stream",
#         },
#     })


# CELL 22 — Quick standalone test (no Flutter needed)

In [ ]:
# Run this cell to verify the transcription pipeline works end-to-end
# before connecting Flutter.
# ════════════════════════════════════════════════════════════════════════════

def _self_test_transcribe(audio_path: str):
    """
    Test the transcription pipeline with a local audio file.
    Pass the path to any .wav / .mp3 / .m4a file.

    Usage in Colab:
        _self_test_transcribe("/content/drive/MyDrive/ZanAI/nena project/test_habari.wav")
    """

    print(f"\n{'─'*60}")
    print(f"  Testing transcription: {audio_path}")
    print(f"{'─'*60}")

    if not os.path.exists(audio_path):
        print(f"  ⚠  File not found: {audio_path}")
        return

    result   = whisper_model.transcribe(audio_path, language="sw", temperature=0.0)
    full_text = result["text"].strip()
    keywords  = extract_keywords(full_text)

    print(f"  Transcript : {full_text}")
    print(f"  Keywords   : {keywords}")
    print(f"  Language   : {result.get('language', 'sw')}")
    print(f"  → Avatar will sign: {' → '.join(keywords) or '(no keywords)'}")
    print(f"{'─'*60}\n")


# Test with a known audio file:
_self_test_transcribe("/content/drive/MyDrive/ZanAI/nena project/test_habari.wav")


────────────────────────────────────────────────────────────
  Testing transcription: /content/drive/MyDrive/ZanAI/nena project/test_habari.wav
────────────────────────────────────────────────────────────
  Transcript : Haba adi.
  Keywords   : ['haba', 'adi']
  Language   : sw
  → Avatar will sign: haba → adi
────────────────────────────────────────────────────────────



# CELL 23 — Summary: what to update in your Flutter dart files

In [ ]:
# After running cells B–F and restarting the Flask server, update:
#
#  1. lib/services/whisper_service.dart
#     Change:  static const String _baseUrl = 'https://YOUR_COLAB_NGROK_URL_HERE';
#     To:      static const String _baseUrl = 'https://<your-new-ngrok-url>';
#
#  2. lib/screens/avatar_screen.dart  (optional enhancement)
#     The current _onWordReceived() takes the first keyword only.
#     To sign ALL keywords in sequence, replace lines 60-69 with:
#
#        void _onWordReceived(String sentence) {
#          if (!mounted) return;
#          final keywords = _whisperService.extractKeywords(sentence);
#          if (keywords.isEmpty) return;
#
#          // Sign each keyword in sequence with a delay
#          Future<void> _signSequence() async {
#            for (final word in keywords) {
#              if (!mounted) return;
#              setState(() {
#                _currentWord = word;
#                _wordHistory.insert(0, word);
#                if (_wordHistory.length > 30) _wordHistory.removeLast();
#              });
#              _flashController.forward(from: 0);
#              await Future.delayed(const Duration(milliseconds: 1200));
#            }
#          }
#          _signSequence();
#        }
#
#  3. No changes needed to avatar_widget.dart — it already handles
#     all 14 sign words plus a 'default' fallback rest position.
#     To add more signs, extend the _signPositions map with new
#     [leftX, leftY, rightX, rightY, bodyTilt] entries.
# ════════════════════════════════════════════════════════════════════════════

print("✅ NENA Speech-to-Avatar backend cells loaded successfully.")
print("   Endpoints registered:")
print("     POST /transcribe         → single-response transcription")
print("     POST /transcribe_stream  → SSE keyword stream")
print()
print("   Next step: restart your Flask server (re-run the start_server cell)")
print("   then update _baseUrl in lib/services/whisper_service.dart.")

✅ NENA Speech-to-Avatar backend cells loaded successfully.
   Endpoints registered:
     POST /transcribe         → single-response transcription
     POST /transcribe_stream  → SSE keyword stream

   Next step: restart your Flask server (re-run the start_server cell)
   then update _baseUrl in lib/services/whisper_service.dart.
